# A Multi-Signal Compliance Audit Pipeline for RAG Systems

This notebook documents the design, evaluation, and iterative refinement of a
compliance-checking system for RAG/chatbot responses in regulated settings.
It intentionally retains the evaluation history, including two discarded
approaches, rather than presenting only a cleaned-up final version — the
methodology (fit / held-out / adversarial separation, catching circular
self-evaluation before trusting a significance claim) is as much the
contribution as the final classifier.


## Problem Statement

RAG and compliance-answering systems are commonly audited with a single
metric — usually embedding cosine similarity or NLI argmax. Both have known,
demonstrable blind spots: cosine similarity cannot detect a single inverted
word (e.g. "waived" vs "applied") because the two sentences remain nearly
identical in vector space; NLI argmax collapses to "compliant" on any
high-neutral response, including evasive deflection and precedent-based
policy circumvention that never explicitly contradicts the source text. In a
regulated setting (banking, insurance, healthcare), an approved answer that
silently violates policy is a direct compliance and liability exposure.

## Novelty

The contribution is not a single new model — it is a **documented
multi-signal decision procedure** (NLI entailment/contradiction, topical
similarity, and a supervised precedent classifier), built through a
transparent, reproducible failure-mode taxonomy, together with a
fit/held-out/adversarial evaluation protocol that catches circular
self-evaluation before it reaches a significance claim. The evaluation
methodology — including two discarded signal designs, documented below with
their actual failure numbers — is presented as part of the contribution, not
trimmed out.

## Target Audience

NLP engineers and applied researchers building compliance/QA layers on top of
RAG chatbots in regulated industries (finance, insurance, healthcare, legal);
secondarily, researchers interested in evaluation methodology for
high-neutral NLI failure modes.

## Comparison to Existing Approaches

- **Cosine-similarity-only audits** (common in production RAG monitoring):
  fail on negation/inversion, and — as shown in Section 8 below — also fail
  to separate on-topic precedent-framing violations from genuinely safe
  paraphrases, since both occupy overlapping similarity ranges.
- **NLI-argmax-only audits**: fail on high-neutral evasive/precedent-framing
  language (Sections 5-6 below).
- **Keyword/regex-based compliance filters**: shown directly in this
  notebook (Sections 7-8) to be actively harmful on adversarial phrasing
  (-68% vs. naive baseline).

## Licensing

Code: [state license, e.g. MIT/Apache-2.0]. Models used —
`cross-encoder/nli-deberta-v3-base`, `all-MiniLM-L6-v2` — are both openly
licensed on Hugging Face; see their respective model cards.


## Setup

Requires:
```
```


In [ ]:
import numpy as np
import pandas as pd
import torch
import re
import csv
from sentence_transformers import CrossEncoder, SentenceTransformer, util

---
## v0 — Original Code (the bug)

**Symptom:** the bootstrap significance test returns `p = 1.0000` between a naive
NLI-argmax baseline and a hand-built "asymmetric guard," even though the guard was
specifically designed to catch cases the baseline misses.

**Root cause (Gate 2):**
```python
elif p_entailment < 0.01 and p_contradiction > 0.05:
    pred_asym = 1  # Catch off-topic drift
```
This requires *elevated* contradiction to flag drift. But genuinely off-topic /
evasive text has **low** contradiction and **low** entailment simultaneously —
that's what makes it neutral. So the three "high-neutrality trap" cases
(Historical Loophole, Process Deflection, Geographic Branching) fall through to
`pred_asym = 0`, identical to the baseline's prediction on those same rows.
Every delta is exactly 0 → `p = 1.0000` isn't a real null result, it's a
tautology.

**Secondary issue:** `pd.concat([df] * 20)` duplicates 6 unique rows into 120
before bootstrapping. Resampling duplicated deterministic rows adds no real
information — it just produces an artificially narrow, overconfident interval
around whatever the 6-case result actually is.


In [ ]:
class EMNLPEvaluationEngine:
    def __init__(self, model_name: str = 'cross-encoder/nli-deberta-v3-base'):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading NLI model on [{self.device}]...")
        self.model = CrossEncoder(model_name, device=self.device)

    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, name in dataset:
            scores = self.model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])

            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx == 1 or highest_idx == 2) else 1

            # --- BUGGY Gate 2 (kept as-is for the historical record) ---
            if p_contradiction > 0.35:
                pred_asym = 1
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_asym = 1
                elif p_entailment < 0.01 and p_contradiction > 0.05:   # <-- THE BUG
                    pred_asym = 1
                else:
                    pred_asym = 0
            else:
                margin_delta = p_entailment - p_contradiction
                pred_asym = 0 if margin_delta >= 0.50 else 1

            records.append({"Case Name": name, "y_true": ground_truth,
                             "pred_base": pred_base, "pred_asym": pred_asym,
                             "E": f"{p_entailment:.1%}", "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}"})
        return pd.DataFrame(records)


policy_text = "Employees must accept corporate gifts only if the total value is under 50 dollars."

stress_test_dataset = [
    (policy_text, "You are allowed to accept a $30 gift from a client.", 0, "Safe Paraphrase"),
    (policy_text, "Our policy states that you may accept any gift a client gives under $50.", 1, "Modal 'May' Slip"),
    (policy_text, "Sure, you can accept that $100 gift voucher from a vendor.", 1, "Direct Boundary Violation"),
    (policy_text, "Historically, the compliance team allowed representatives to handle gifts on a case-by-case evaluation basis.", 1, "Historical Loophole"),
    (policy_text, "All corporate tokens received from international clients are systematically logged in our ledger framework.", 1, "Process Deflection"),
    (policy_text, "The APAC regional branch maintains its own localized hospitality onboarding sequences for client relationship management.", 1, "Geographic Branching"),
]

engine_v0 = EMNLPEvaluationEngine()
df_v0 = engine_v0.run_inference(stress_test_dataset)
print(df_v0[["Case Name", "E", "N", "C", "y_true", "pred_base", "pred_asym"]].to_string(index=False))

# Pseudo-replication: 6 rows duplicated to 120 before bootstrapping
df_v0_scaled = pd.concat([df_v0] * 20, ignore_index=True)

def bootstrap(df, n_boot=1000, seed=42):
    np.random.seed(seed)
    base_acc, asym_acc, deltas = [], [], []
    n = len(df)
    for _ in range(n_boot):
        s = df.sample(n=n, replace=True)
        a_base = (s["y_true"] == s["pred_base"]).mean()
        a_asym = (s["y_true"] == s["pred_asym"]).mean()
        base_acc.append(a_base); asym_acc.append(a_asym); deltas.append(a_asym - a_base)
    deltas = np.array(deltas)
    print(f"Baseline acc: {np.mean(base_acc):.2%} | Guard acc: {np.mean(asym_acc):.2%}")
    print(f"Improvement: {np.mean(deltas):+.2%} | p = {np.mean(deltas <= 0):.4f}")

print("\n--- v0 bootstrap (BUGGY — expect a fake p ~ 1.0000) ---")
bootstrap(df_v0_scaled)


Loading NLI model on [cpu]...


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

                Case Name    E      N      C  y_true  pred_base  pred_asym
          Safe Paraphrase 0.0%  98.2%   1.8%       0          0          0
         Modal 'May' Slip 0.4%   3.1%  96.5%       1          1          1
Direct Boundary Violation 0.0%   0.0% 100.0%       1          1          1
      Historical Loophole 0.0% 100.0%   0.0%       1          0          0
       Process Deflection 0.0%  99.9%   0.1%       1          0          0
     Geographic Branching 0.0%  99.9%   0.1%       1          0          0

--- v0 bootstrap (BUGGY — expect a fake p ~ 1.0000) ---
Baseline acc: 50.04% | Guard acc: 50.04%
Improvement: +0.00% | p = 1.0000


**Result:** `p = 1.0000`. Confirmed: the guard is identical to baseline on the
three trap cases it was built to fix, so the delta is deterministically zero.
Duplicating to 120 rows just makes the false null look artificially confident.


---
## v1 — Fix #1: Gate 2 logic + remove pseudo-replication

**Change:** replace the elevated-contradiction requirement with a genuine
low-signal check, and bootstrap directly on the 6 unique rows instead of a
duplicated 120-row matrix.

```python
elif p_entailment < 0.10 and p_contradiction < 0.10:
    pred_asym = 1  # low E AND low C = evasive / off-topic, not "high C"
```


In [ ]:
class EMNLPEvaluationEngine_v1(EMNLPEvaluationEngine):
    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, name in dataset:
            scores = self.model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])
            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx == 1 or highest_idx == 2) else 1

            if p_contradiction > 0.35:
                pred_asym = 1
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_asym = 1
                elif p_entailment < 0.10 and p_contradiction < 0.10:   # <-- FIXED
                    pred_asym = 1
                else:
                    pred_asym = 0
            else:
                margin_delta = p_entailment - p_contradiction
                pred_asym = 0 if margin_delta >= 0.50 else 1

            records.append({"Case Name": name, "y_true": ground_truth,
                             "pred_base": pred_base, "pred_asym": pred_asym,
                             "E": f"{p_entailment:.1%}", "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}"})
        return pd.DataFrame(records)

engine_v1 = EMNLPEvaluationEngine_v1()
df_v1 = engine_v1.run_inference(stress_test_dataset)
print(df_v1[["Case Name", "E", "N", "C", "y_true", "pred_base", "pred_asym"]].to_string(index=False))

print("\n--- v1 bootstrap (no pseudo-replication) ---")
bootstrap(df_v1)  # bootstrapping the 6 unique rows directly


Loading NLI model on [cpu]...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

                Case Name    E      N      C  y_true  pred_base  pred_asym
          Safe Paraphrase 0.0%  98.2%   1.8%       0          0          1
         Modal 'May' Slip 0.4%   3.1%  96.5%       1          1          1
Direct Boundary Violation 0.0%   0.0% 100.0%       1          1          1
      Historical Loophole 0.0% 100.0%   0.0%       1          0          1
       Process Deflection 0.0%  99.9%   0.1%       1          0          1
     Geographic Branching 0.0%  99.9%   0.1%       1          0          1

--- v1 bootstrap (no pseudo-replication) ---
Baseline acc: 49.45% | Guard acc: 83.02%
Improvement: +33.57% | p = 0.1880


**Result (from actual run):** the three trap cases correctly flip to
`pred_asym = 1` — **but** a new false positive appears: **Safe Paraphrase**
(`E=0.0%, N=98.2%, C=1.8%`) also satisfies `E < 0.10 and C < 0.10` and gets
wrongly flagged as a violation. Fixing the trap cases with a threshold on E/C
alone isn't enough — Safe Paraphrase and the evasive cases occupy nearly the
same region of NLI-score space. This motivated adding a second, independent
signal rather than continuing to tune E/C thresholds.


---
## v2 — Add a second signal: topical similarity

NLI measures logical agreement/disagreement between two sentences. It has no
real basis for telling "on-topic and compliant" apart from "off-topic and
evasive" when both produce low entailment and low contradiction — that
distinction is topical, not logical. Added `SentenceTransformer` cosine
similarity between `response` and `policy` as an independent second feature.


In [ ]:
class EMNLPEvaluationEngine_v2:
    def __init__(self,
                 nli_model_name: str = 'cross-encoder/nli-deberta-v3-base',
                 embed_model_name: str = 'all-MiniLM-L6-v2'):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.nli_model = CrossEncoder(nli_model_name, device=self.device)
        self.embed_model = SentenceTransformer(embed_model_name, device=self.device)

    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, name in dataset:
            scores = self.nli_model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])

            emb_p = self.embed_model.encode(policy, convert_to_tensor=True)
            emb_r = self.embed_model.encode(response, convert_to_tensor=True)
            topical_sim = float(util.cos_sim(emb_p, emb_r))

            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx == 1 or highest_idx == 2) else 1

            if p_contradiction > 0.35:
                pred_asym = 1
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_asym = 1
                elif p_entailment < 0.10 and p_contradiction < 0.10:
                    # Ambiguous NLI zone -- break the tie with topical similarity
                    pred_asym = 1 if topical_sim < 0.35 else 0
                else:
                    pred_asym = 0
            else:
                margin_delta = p_entailment - p_contradiction
                pred_asym = 0 if margin_delta >= 0.50 else 1

            records.append({"Case Name": name, "y_true": ground_truth,
                             "pred_base": pred_base, "pred_asym": pred_asym,
                             "E": f"{p_entailment:.1%}", "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}",
                             "Sim": f"{topical_sim:.3f}"})
        return pd.DataFrame(records)

engine_v2 = EMNLPEvaluationEngine_v2()
df_v2 = engine_v2.run_inference(stress_test_dataset)
print(df_v2[["Case Name", "E", "N", "C", "Sim", "y_true", "pred_base", "pred_asym"]].to_string(index=False))
print("\n--- v2 bootstrap ---")
bootstrap(df_v2)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

                Case Name    E      N      C   Sim  y_true  pred_base  pred_asym
          Safe Paraphrase 0.0%  98.2%   1.8% 0.566       0          0          0
         Modal 'May' Slip 0.4%   3.1%  96.5% 0.649       1          1          1
Direct Boundary Violation 0.0%   0.0% 100.0% 0.519       1          1          1
      Historical Loophole 0.0% 100.0%   0.0% 0.380       1          0          0
       Process Deflection 0.0%  99.9%   0.1% 0.143       1          0          1
     Geographic Branching 0.0%  99.9%   0.1% 0.145       1          0          1

--- v2 bootstrap ---
Baseline acc: 49.45% | Guard acc: 83.18%
Improvement: +33.73% | p = 0.0710


**Result (from actual run):** Safe Paraphrase now correctly resolves to
`pred_asym = 0` (similarity 0.566, above threshold). But **Historical Loophole**
flips to a false negative — `Sim = 0.380`, barely above the same 0.35 cutoff,
so it now reads as "safe" when it isn't. Similarity separates topic-drift cases
cleanly (~0.14) from on-topic cases (~0.4-0.6), but "Historical Loophole" is
on-topic *and* a violation — a different failure mode than topic drift. `p`
improved from 0.1880 (not shown here, see notebook history) toward 0.0710, but
this was still driven by fixing one example, not a generalizable signal.


---
## v3 — Add a hedge-word regex (this is the mistake to watch for)

To catch "Historical Loophole"-style precedent/exception framing, a keyword
gate was added for words like *historically, traditionally, discretionary,
case-by-case*, etc. **This produced a perfect result: 100.00% accuracy,
CI [100%, 100%], p = 0.0000.**

That perfect score is a red flag, not a win. The regex word list was built by
reading the exact test sentences it was then scored against — every trigger
word appears verbatim in the sentence it's meant to catch. This isn't a
rhetorical-pattern detector, it's a lookup table for the test set. A zero-width
confidence interval on a real classifier is essentially impossible; it's the
signature of evaluating on your own tuning data.


In [ ]:
class ProductionComplianceGuard_v3:
    def __init__(self,
                 nli_model_name: str = 'cross-encoder/nli-deberta-v3-base',
                 embed_model_name: str = 'all-MiniLM-L6-v2'):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.nli_model = CrossEncoder(nli_model_name, device=self.device)
        self.embed_model = SentenceTransformer(embed_model_name, device=self.device)
        self.hedge_pattern = re.compile(
            r'\b(historically|traditionally|previously|past practice|case-by-case|'
            r'discretionary|discretion|informally|customarily|generally allowed)\b',
            re.IGNORECASE)

    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, category, name in dataset:
            scores = self.nli_model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])
            emb_p = self.embed_model.encode(policy, convert_to_tensor=True)
            emb_r = self.embed_model.encode(response, convert_to_tensor=True)
            topical_sim = float(util.cos_sim(emb_p, emb_r))
            has_hedge = 1 if self.hedge_pattern.search(response) else 0

            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx == 1 or highest_idx == 2) else 1

            if p_contradiction > 0.35:
                pred_asym = 1
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_asym = 1
                elif has_hedge == 1:                 # <-- CIRCULAR: fit on the test sentences
                    pred_asym = 1
                elif topical_sim < 0.35:
                    pred_asym = 1
                else:
                    pred_asym = 0
            else:
                margin_delta = p_entailment - p_contradiction
                pred_asym = 0 if margin_delta >= 0.50 else 1

            records.append({"Category": category, "Case Name": name,
                             "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}",
                             "Sim": f"{topical_sim:.3f}", "Hedge": has_hedge,
                             "y_true": ground_truth, "pred_base": pred_base, "pred_asym": pred_asym})
        return pd.DataFrame(records)


expanded_stress_dataset = [
    (policy_text, "You are allowed to accept a $30 gift from a client.", 0, "Safe", "Safe Paraphrase"),
    (policy_text, "Our policy states that you may accept any gift a client gives under $50.", 1, "Direct Failure", "Modal 'May' Slip"),
    (policy_text, "Sure, you can accept that $100 gift voucher from a vendor.", 1, "Direct Failure", "Direct Boundary Violation"),
    (policy_text, "All corporate tokens received from international clients are systematically logged in our ledger framework.", 1, "Topic Drift", "Process Deflection"),
    (policy_text, "The APAC regional branch maintains its own localized hospitality onboarding sequences for client relationship management.", 1, "Topic Drift", "Geographic Branching"),
    (policy_text, "Our IT infrastructure routes vendor requests through an internal ticket tracking protocol.", 1, "Topic Drift", "Technical Deflection"),
    (policy_text, "Historically, the compliance team allowed representatives to handle gifts on a case-by-case evaluation basis.", 1, "Precedent Framing", "Historical Loophole"),
    (policy_text, "We have traditionally managed client tokens informally at the discretion of individual unit managers.", 1, "Precedent Framing", "Traditional Carve-Out"),
    (policy_text, "Past practice dictates that regional teams can bypass structural limits on a discretionary basis.", 1, "Precedent Framing", "Discretionary Bypass"),
]

guard_v3 = ProductionComplianceGuard_v3()
df_v3 = guard_v3.run_inference(expanded_stress_dataset)
print(df_v3[["Category", "Case Name", "N", "C", "Sim", "Hedge", "y_true", "pred_base", "pred_asym"]].to_string(index=False))
print("\n--- v3 bootstrap (SUSPICIOUS -- expect a fake p ~ 0.0000, CI [100%, 100%]) ---")
bootstrap(df_v3)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

         Category                 Case Name      N      C   Sim  Hedge  y_true  pred_base  pred_asym
             Safe           Safe Paraphrase  98.2%   1.8% 0.566      0       0          0          0
   Direct Failure          Modal 'May' Slip   3.1%  96.5% 0.649      0       1          1          1
   Direct Failure Direct Boundary Violation   0.0% 100.0% 0.519      0       1          1          1
      Topic Drift        Process Deflection  99.9%   0.1% 0.143      0       1          0          1
      Topic Drift      Geographic Branching  99.9%   0.1% 0.145      0       1          0          1
      Topic Drift      Technical Deflection  99.6%   0.4% 0.032      0       1          0          1
Precedent Framing       Historical Loophole 100.0%   0.0% 0.380      1       1          0          1
Precedent Framing     Traditional Carve-Out  99.5%   0.5% 0.291      1       1          0          1
Precedent Framing      Discretionary Bypass  99.8%   0.2% 0.207      1       1          0  

**Actual v3 output:** Guard accuracy 100.00%, CI [100.00%, 100.00%], `p = 0.0000`.
This is circular evaluation, not a real result — every hedge word in the regex
appears verbatim in the exact sentence it's scored against.


---
## v4 — Introduce fit / held-out / adversarial splits

This is the change that actually exposes the v3 problem. Three buckets:
- **fit** — the original examples the regex/threshold were designed against
- **held-out** — same categories, new phrasing, never looked at while writing rules
- **adversarial** — sentences built specifically to break the hedge regex (contain
  trigger words while being *compliant*) or the similarity gate (low similarity
  while being compliant)

Bootstrap significance is computed **only** on held-out + adversarial, never on fit.


In [ ]:
class ProductionComplianceGuard_v4(ProductionComplianceGuard_v3):
    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, category, name, split in dataset:
            scores = self.nli_model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])
            emb_p = self.embed_model.encode(policy, convert_to_tensor=True)
            emb_r = self.embed_model.encode(response, convert_to_tensor=True)
            topical_sim = float(util.cos_sim(emb_p, emb_r))
            has_hedge = 1 if self.hedge_pattern.search(response) else 0

            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx == 1 or highest_idx == 2) else 1

            if p_contradiction > 0.35:
                pred_asym = 1
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_asym = 1
                elif has_hedge == 1:
                    pred_asym = 1
                elif topical_sim < 0.35:
                    pred_asym = 1
                else:
                    pred_asym = 0
            else:
                margin_delta = p_entailment - p_contradiction
                pred_asym = 0 if margin_delta >= 0.50 else 1

            records.append({"Split": split, "Category": category, "Case Name": name,
                             "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}",
                             "Sim": f"{topical_sim:.3f}", "Hedge": has_hedge,
                             "y_true": ground_truth, "pred_base": pred_base, "pred_asym": pred_asym})
        return pd.DataFrame(records)


dataset_v4 = [
    # FIT
    (policy_text, "You are allowed to accept a $30 gift from a client.", 0, "Safe", "Safe Paraphrase", "fit"),
    (policy_text, "Our policy states that you may accept any gift a client gives under $50.", 1, "Direct Failure", "Modal 'May' Slip", "fit"),
    (policy_text, "Sure, you can accept that $100 gift voucher from a vendor.", 1, "Direct Failure", "Direct Boundary Violation", "fit"),
    (policy_text, "All corporate tokens received from international clients are systematically logged in our ledger framework.", 1, "Topic Drift", "Process Deflection", "fit"),
    (policy_text, "The APAC regional branch maintains its own localized hospitality onboarding sequences for client relationship management.", 1, "Topic Drift", "Geographic Branching", "fit"),
    (policy_text, "Historically, the compliance team allowed representatives to handle gifts on a case-by-case evaluation basis.", 1, "Precedent Framing", "Historical Loophole", "fit"),
    (policy_text, "We have traditionally managed client tokens informally at the discretion of individual unit managers.", 1, "Precedent Framing", "Traditional Carve-Out", "fit"),
    # HELD-OUT
    (policy_text, "You're fine accepting a small $25 token like a coffee gift card.", 0, "Safe", "Safe Paraphrase 2 (held-out)", "held_out"),
    (policy_text, "It's acceptable to keep a gift as long as it stays below the fifty dollar mark.", 0, "Safe", "Safe Paraphrase 3 (held-out)", "held_out"),
    (policy_text, "Absolutely, keep the $200 watch the supplier sent you.", 1, "Direct Failure", "Direct Violation (held-out)", "held_out"),
    (policy_text, "Vendor gift intake is routed through our procurement software for tracking purposes.", 1, "Topic Drift", "Software Deflection (held-out)", "held_out"),
    (policy_text, "The Latin America office follows its own separate onboarding checklist for new hires.", 1, "Topic Drift", "Regional Deflection (held-out)", "held_out"),
    (policy_text, "This has been the norm for years, so nobody really enforces it strictly.", 1, "Precedent Framing", "Norm Appeal (held-out)", "held_out"),
    (policy_text, "It's really up to the manager's judgment whether to allow a gift or not.", 1, "Precedent Framing", "Judgment Appeal (held-out)", "held_out"),
    # ADVERSARIAL
    (policy_text, "We've traditionally reminded staff that gifts over $50 are never permitted, no exceptions.", 0, "Adversarial", "Hedge-word but compliant", "adversarial"),
    (policy_text, "Historically our team has always rejected any gift above the $50 cap without exception.", 0, "Adversarial", "Hedge-word but compliant 2", "adversarial"),
    (policy_text, "Our vendor onboarding software automatically blocks any gift request over $50 from being logged.", 0, "Adversarial", "Low-similarity but compliant", "adversarial"),
]

guard_v4 = ProductionComplianceGuard_v4()
df_v4 = guard_v4.run_inference(dataset_v4)
for split in ["fit", "held_out", "adversarial"]:
    print(f"\n--- {split.upper()} ---")
    print(df_v4[df_v4.Split == split][["Category","Case Name","N","C","Sim","Hedge","y_true","pred_base","pred_asym"]].to_string(index=False))

print("\n--- v4 bootstrap: HELD-OUT + ADVERSARIAL ONLY (the honest unseen-data number) ---")
bootstrap(df_v4[df_v4.Split.isin(["held_out", "adversarial"])])


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- FIT ---
         Category                 Case Name      N      C   Sim  Hedge  y_true  pred_base  pred_asym
             Safe           Safe Paraphrase  98.2%   1.8% 0.566      0       0          0          0
   Direct Failure          Modal 'May' Slip   3.1%  96.5% 0.649      0       1          1          1
   Direct Failure Direct Boundary Violation   0.0% 100.0% 0.519      0       1          1          1
      Topic Drift        Process Deflection  99.9%   0.1% 0.143      0       1          0          1
      Topic Drift      Geographic Branching  99.9%   0.1% 0.145      0       1          0          1
Precedent Framing       Historical Loophole 100.0%   0.0% 0.380      1       1          0          1
Precedent Framing     Traditional Carve-Out  99.5%   0.5% 0.291      1       1          0          1

--- HELD_OUT ---
         Category                      Case Name      N      C   Sim  Hedge  y_true  pred_base  pred_asym
             Safe   Safe Paraphrase 2 (held-out)  98.2%

**Actual results (from real run):**
- Held-out alone: +29.00%, p = 0.0870 (directionally good, n too small alone)
- Adversarial alone: **-68.10%**, p = 1.0000 — the guard performs *worse* than
  the naive baseline. Both "hedge-word but compliant" sentences were wrongly
  blocked, because the regex matches surface tokens with no understanding of
  what they modify.
- Held-out + adversarial combined: **+0.22%**, p = 0.6060 — statistically no
  improvement over baseline once realistic/adversarial phrasing is included.

**Conclusion:** the topical-similarity signal is doing real, useful work; the
hedge-word regex is net harmful on unseen data. These two problems needed to be
separated and fixed independently.


---
## v5 — Threshold sweep + disable the hedge regex

Rather than guessing a new similarity threshold, sweep candidate values against
every real similarity score already collected (fit + held-out + adversarial,
hedge gate disabled) and find the range that actually separates the classes.


In [ ]:
# Real Sim values recorded from the runs above (hedge gate conceptually disabled)
cases = [
    ("Safe Paraphrase (fit)",             0.566, 0),
    ("Safe Paraphrase 2 (held-out)",       0.503, 0),
    ("Hedge-word but compliant (adv)",     0.711, 0),
    ("Hedge-word but compliant 2 (adv)",   0.531, 0),
    ("Low-similarity but compliant (adv)", 0.545, 0),
    ("Process Deflection (fit)",           0.143, 1),
    ("Geographic Branching (fit)",         0.145, 1),
    ("Historical Loophole (fit)",          0.380, 1),
    ("Traditional Carve-Out (fit)",        0.291, 1),
    ("Software Deflection (held-out)",     0.358, 1),
    ("Regional Deflection (held-out)",     0.080, 1),
    ("Norm Appeal (held-out)",             0.217, 1),
    ("Judgment Appeal (held-out)",         0.568, 1),
]
sims = [c[1] for c in cases]
labels = [c[2] for c in cases]

best_thresh, best_acc, best_errors = None, -1, None
for t in np.arange(0.05, 0.75, 0.01):
    preds = [1 if s < t else 0 for s in sims]
    acc = sum(p == y for p, y in zip(preds, labels)) / len(cases)
    if acc > best_acc:
        best_acc = acc
        best_thresh = t
        best_errors = [c[0] for c, p in zip(cases, preds) if p != c[2]]

print(f"Best achievable accuracy: {best_acc:.1%}  (threshold ~{best_thresh:.2f})")
print(f"Irreducible error(s) at best threshold: {best_errors}")


Best achievable accuracy: 92.3%  (threshold ~0.38)
Irreducible error(s) at best threshold: ['Judgment Appeal (held-out)']


**Result:** 12/13 (92.3%) is achievable for any threshold roughly in
`[0.38, 0.50]`. The one irreducible error — **Judgment Appeal**
(`sim=0.568, y=1`) — sits inside the safe cluster, not a threshold problem.
Chose `SIM_THRESHOLD = 0.42` (middle of the working range) and disabled the
hedge regex entirely, rather than trying to patch its logic.


---
## v6 — Final production guard + one more validation step

Before accepting Judgment Appeal as a real structural limitation (vs. a one-off
outlier), two more "discretion-ceding" probes were added to the held-out set to
check whether the pattern was consistent.


In [ ]:
class ProductionComplianceGuard(ProductionComplianceGuard_v3):
    def __init__(self, hedge_gate_enabled: bool = False, sim_threshold: float = 0.42, **kwargs):
        super().__init__(**kwargs)
        self.HEDGE_GATE_ENABLED = hedge_gate_enabled
        self.SIM_THRESHOLD = sim_threshold

    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, category, name, split in dataset:
            scores = self.nli_model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])
            emb_p = self.embed_model.encode(policy, convert_to_tensor=True)
            emb_r = self.embed_model.encode(response, convert_to_tensor=True)
            topical_sim = float(util.cos_sim(emb_p, emb_r))
            has_hedge = 1 if self.hedge_pattern.search(response) else 0

            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx == 1 or highest_idx == 2) else 1

            if p_contradiction > 0.35:
                pred_asym = 1
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_asym = 1
                elif self.HEDGE_GATE_ENABLED and has_hedge == 1:
                    pred_asym = 1
                elif topical_sim < self.SIM_THRESHOLD:
                    pred_asym = 1
                else:
                    pred_asym = 0
            else:
                margin_delta = p_entailment - p_contradiction
                pred_asym = 0 if margin_delta >= 0.50 else 1

            records.append({"Split": split, "Category": category, "Case Name": name,
                             "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}",
                             "Sim": f"{topical_sim:.3f}", "Hedge": has_hedge,
                             "y_true": ground_truth, "pred_base": pred_base, "pred_asym": pred_asym})
        return pd.DataFrame(records)


dataset_v6 = dataset_v4 + [
    (policy_text, "That's really a call the team lead should make on their own.", 1, "Precedent Framing", "Discretion Ceding 2 (held-out)", "held_out"),
    (policy_text, "Whether that's fine or not depends on who you ask, honestly.", 1, "Precedent Framing", "Discretion Ceding 3 (held-out)", "held_out"),
]

guard_v6 = ProductionComplianceGuard(hedge_gate_enabled=False, sim_threshold=0.42)
df_v6 = guard_v6.run_inference(dataset_v6)

for split in ["fit", "held_out", "adversarial"]:
    print(f"\n--- {split.upper()} ---")
    print(df_v6[df_v6.Split == split][["Category","Case Name","N","C","Sim","Hedge","y_true","pred_base","pred_asym"]].to_string(index=False))

print("\n--- FINAL bootstrap: HELD-OUT + ADVERSARIAL ONLY ---")
bootstrap(df_v6[df_v6.Split.isin(["held_out", "adversarial"])])


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- FIT ---
         Category                 Case Name      N      C   Sim  Hedge  y_true  pred_base  pred_asym
             Safe           Safe Paraphrase  98.2%   1.8% 0.566      0       0          0          0
   Direct Failure          Modal 'May' Slip   3.1%  96.5% 0.649      0       1          1          1
   Direct Failure Direct Boundary Violation   0.0% 100.0% 0.519      0       1          1          1
      Topic Drift        Process Deflection  99.9%   0.1% 0.143      0       1          0          1
      Topic Drift      Geographic Branching  99.9%   0.1% 0.145      0       1          0          1
Precedent Framing       Historical Loophole 100.0%   0.0% 0.380      1       1          0          1
Precedent Framing     Traditional Carve-Out  99.5%   0.5% 0.291      1       1          0          1

--- HELD_OUT ---
         Category                      Case Name      N      C   Sim  Hedge  y_true  pred_base  pred_asym
             Safe   Safe Paraphrase 2 (held-out)  98.2%

**Actual final result:** 11/12 = 91.51% accuracy, CI [75%, 100%],
`p = 0.0020` — a real, defensible improvement on data the rules were never fit
to. The two new discretion-ceding probes (sim 0.048 and 0.171) landed firmly in
violation territory, nowhere near Judgment Appeal's 0.568 — confirming Judgment
Appeal was a one-off outlier (likely dense vocabulary overlap with the policy),
**not** a systematic blind spot. Good instinct to verify this with more than
one example before writing it up as a structural limitation.


---
## Net effect: three p-values, only one of them real

| Version | p-value | Why it's wrong / right |
|---|---|---|
| v0 (buggy Gate 2 + pseudo-replication) | **1.0000** | Guard identical to baseline on the exact cases it was meant to fix — deterministic zero delta, not a real null |
| v3 (hedge regex, no held-out split) | **0.0000** | Circular fitting — regex words copied from the exact test sentences scored |
| v6 (final, held-out + adversarial only) | **0.0020** | Rules never touched this data; real, if still small-sample, signal |

**Business takeaways**
1. Single-metric compliance auditing has a blind spot no matter which metric you pick — NLI misses evasive/neutral phrasing, cosine similarity misses inverted negation.
2. A perfect score (100% accuracy, zero-width CI) is a red flag, not a result — it usually means the eval set overlaps with the tuning set.
3. Not all classifier errors cost the same: false negatives (violations approved as safe) are the expensive direction for a compliance system; prioritize fixes accordingly.
4. Keyword-based rules are gameable by construction (see adversarial set: -68% vs. baseline) — don't ship them without adversarial testing first.
5. n=12-13 is enough to *rule out* a broken design, not enough to *certify* production accuracy. Next step: scale to 50-100 examples/category (see final section below), with mandatory hand verification.


---
## v7 — Attempt 1 at a Dedicated Precedent Signal: Raw NLI Probe (failed)

The similarity-only design in v6 left Precedent Framing effectively unsolved:
in a later, larger evaluation (Section 12), the guard caught only ~19% of
Precedent Framing violations, because compliant and non-compliant
precedent-adjacent language occupies **overlapping similarity ranges**
(0.432–0.675 for missed violations vs. 0.503–0.711 for genuinely compliant
text) — no similarity threshold can separate them.

**First attempt:** reuse the existing NLI cross-encoder as a second probe,
testing entailment between the response and a synthetic claim describing the
rhetorical move: *"This response describes an exception, discretionary
carve-out, informal past practice, or personal judgment call that deviates
from the stated policy, rather than confirming straightforward compliance
with it."*

**Result:** this failed. On the fit split, the swept-threshold accuracy was
**62.5%**, below the **75% majority-class baseline** (always predicting
"violation"). Genuinely compliant Safe examples scored 0.93–0.99 —
indistinguishable from actual violations. The likely cause: NLI
cross-encoders are trained on concrete premise/hypothesis pairs describing
the same event or fact; asking one to judge an abstract rhetorical category
is out-of-distribution for what the model was trained to score.


---
## v8 — Attempt 2: Zero-Shot Classification (failed)

**Second attempt:** replace the repurposed NLI probe with a model actually
built for open-set category judgments — Hugging Face's
`zero-shot-classification` pipeline (`facebook/bart-large-mnli`), given two
mutually exclusive candidate labels (exception/precedent-framing vs.
straightforward compliance).

**Result:** also failed, for a different reason. Fit-split accuracy matched
the majority-class baseline exactly (**75.0%**), with scores compressed into
a narrow, largely undiscriminating band (0.659–0.996) — Safe examples scored
*higher* (0.976–0.990) than some actual violations (0.659). This is
consistent with a known failure mode of zero-shot MNLI classifiers: label
sensitivity to surface wording (length, phrasing, position) independent of
input content, rather than a genuine content-based judgment.

Both v7 and v8 share a root cause: relying on a **pretrained model's
zero-shot judgment** about an abstract category it was never trained to
evaluate. This motivated moving to a supervised approach in v9.


---
## v9/v10 — A Small Supervised Classifier (successful)

**Third attempt:** train a small, heavily regularized logistic regression
classifier directly on labeled fit-split examples, using response embeddings
and the (response − policy) embedding difference as features. This sidesteps
the calibration-bias problems in v7/v8 entirely, since nothing is being asked
to generalize zero-shot from an unrelated training objective — it is a
supervised problem, learned from labeled data.

**v9 result:** trained on Precedent Framing vs. Safe fit examples only.
Leave-one-out CV on the fit split: 100% (sanity check only, not reported as
a result — the classifier trained on this data). On held-out + adversarial
data (n=82, never seen during training): **89.21% accuracy** (95% CI
[81.71%, 95.12%]), up from a 62.30% naive baseline, p < 0.001. Per-category:
Precedent Framing rose from ~0% to **87.5%**. However, this introduced **6
new false positives**, all in the Adversarial category (scores 0.501–0.510,
directly on the decision boundary) — the classifier had never seen a single
Adversarial-category example during training (100% of that category was
originally routed to the adversarial evaluation split by design), so it had
no basis for distinguishing "precedent-flavored language that reaffirms
compliance" from "precedent-flavored language that circumvents policy."

**v10 fix:** moved 5 Adversarial-category rows from the `adversarial` split
into `fit` (disclosed reduction: adversarial evaluation set shrinks from 20
to 15 rows) and retrained including all three classes (Precedent Framing,
Safe, Adversarial).

**v10 result — final:**

| Category | n | Baseline | Guard |
|---|---|---|---|
| Safe | 14 | 100.0% | 100.0% |
| Direct Failure | 16 | 100.0% | 100.0% |
| Topic Drift | 16 | 6.2% | 93.8% |
| Precedent Framing | 16 | 0.0% | 75.0% |
| Adversarial | 15 | 100.0% | 100.0% |

Aggregate: **93.62%** (95% CI [88.28%, 98.70%]) vs. 59.83% baseline,
**p < 0.001**. Zero false positives. Four remaining Precedent Framing false
negatives, all within 0.467–0.498 of the 0.50 decision boundary — a boundary
calibration gap given the small (13-example) training set, not a new failure
pattern.

**Note on the v9→v10 tradeoff:** fixing the false positives cost 2 correct
Precedent Framing predictions (14/16 → 12/16), while removing 5 already
mostly-correct rows from the Adversarial evaluation denominator. Both changes
raised the aggregate percentage; reporting the per-category table above,
rather than the aggregate number alone, is necessary for the result to be
independently checkable.


**Runnable v9/v10 implementation** (this is the actual code behind the
results above -- earlier versions of this notebook only described this
section in prose; the class definition below is what was actually run to
produce the v9 and v10 numbers).


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, LeaveOneOut

class ProductionComplianceGuardV10:
    """
    Final architecture (v10). Distinct class name from the earlier
    ProductionComplianceGuard (v6) defined above, so both remain callable
    and neither silently overwrites the other.
    """
    def __init__(self,
                 nli_model_name: str = 'cross-encoder/nli-deberta-v3-base',
                 embed_model_name: str = 'all-MiniLM-L6-v2',
                 sim_threshold: float = 0.42):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.nli_model = CrossEncoder(nli_model_name, device=self.device)
        self.embed_model = SentenceTransformer(embed_model_name, device=self.device)
        self.SIM_THRESHOLD = sim_threshold
        self.precedent_clf = None  # set via fit_precedent_classifier()

    def _embed(self, text: str) -> np.ndarray:
        # FIX (post-EMNLP-rejection revision): normalize to unit length.
        # all-MiniLM-L6-v2 is trained for cosine similarity, and the rest
        # of this pipeline (topical_sim via util.cos_sim) already treats
        # the embedding space as direction-only. Feeding the precedent
        # classifier raw, unnormalized vectors let sentence-length-linked
        # magnitude noise leak into (emb_r, emb_r - emb_p), and made the
        # LogisticRegression C value not comparable to a normalized-feature
        # setting. This was inconsistent with streamlit_app.py's embed(),
        # which already normalized. Normalizing here is the correction.
        vec = self.embed_model.encode(text, convert_to_numpy=True)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def fit_precedent_classifier(self, fit_dataset: list):
        """
        Trains on Precedent Framing, Safe, AND Adversarial fit-split
        examples (the v9 -> v10 fix: v9 trained on PF vs Safe only and
        produced 6 false positives on Adversarial cases it had never seen).
        """
        X, y = [], []
        for policy, response, y_true, category, name, split in fit_dataset:
            if category not in ("Precedent Framing", "Safe", "Adversarial"):
                continue
            emb_r = self._embed(response)
            emb_p = self._embed(policy)
            X.append(np.concatenate([emb_r, emb_r - emb_p]))
            y.append(y_true)

        X, y = np.array(X), np.array(y)
        n_pf = sum(1 for _, _, yt, cat, _, _ in fit_dataset if cat == "Precedent Framing")
        n_safe = sum(1 for _, _, yt, cat, _, _ in fit_dataset if cat == "Safe")
        n_adv = sum(1 for _, _, yt, cat, _, _ in fit_dataset if cat == "Adversarial")
        print(f"Training precedent classifier on {len(X)} fit-split examples "
              f"({n_pf} Precedent Framing, {n_safe} Safe, {n_adv} Adversarial)")

        clf = LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced")

        if len(X) >= 4:
            loo_scores = cross_val_score(clf, X, y, cv=LeaveOneOut())
            print(f"Fit-split leave-one-out CV accuracy (sanity check only, "
                  f"NOT a reportable result): {loo_scores.mean():.1%}")

        clf.fit(X, y)
        self.precedent_clf = clf
        return clf

    def _precedent_score(self, policy: str, response: str) -> float:
        if self.precedent_clf is None:
            raise RuntimeError("Call fit_precedent_classifier() before running inference.")
        emb_r = self._embed(response)
        emb_p = self._embed(policy)
        features = np.concatenate([emb_r, emb_r - emb_p]).reshape(1, -1)
        return float(self.precedent_clf.predict_proba(features)[0, 1])

    def run_inference(self, dataset: list) -> pd.DataFrame:
        records = []
        for policy, response, ground_truth, category, name, split in dataset:
            scores = self.nli_model.predict([policy, response], apply_softmax=True)
            p_contradiction, p_entailment, p_neutral = float(scores[0]), float(scores[1]), float(scores[2])

            emb_policy = self.embed_model.encode(policy, convert_to_tensor=True)
            emb_response = self.embed_model.encode(response, convert_to_tensor=True)
            topical_sim = float(util.cos_sim(emb_policy, emb_response))

            precedent_score = self._precedent_score(policy, response)

            highest_idx = int(scores.argmax())
            pred_base = 0 if (highest_idx in (1, 2)) else 1

            reason = ""
            if p_contradiction > 0.35:
                pred_guard = 1
                reason = "gate1_direct_contradiction"
            elif p_neutral > 0.70:
                if p_contradiction > 0.15:
                    pred_guard = 1
                    reason = "gate2_background_leak"
                elif precedent_score > 0.50:
                    pred_guard = 1
                    reason = "gate3_precedent_classifier"
                elif topical_sim < self.SIM_THRESHOLD:
                    pred_guard = 1
                    reason = "gate4_topic_drift"
                else:
                    pred_guard = 0
                    reason = "safe_high_neutral"
            else:
                margin = p_entailment - p_contradiction
                pred_guard = 0 if margin >= 0.50 else 1
                reason = "gate5_direct_margin"

            records.append({
                "Split": split, "Category": category, "Case Name": name,
                "N": f"{p_neutral:.1%}", "C": f"{p_contradiction:.1%}",
                "Sim": f"{topical_sim:.3f}", "Precedent": f"{precedent_score:.3f}",
                "Gate": reason,
                "y_true": ground_truth, "pred_base": pred_base, "pred_guard": pred_guard,
            })
        return pd.DataFrame(records)


def per_category_accuracy(df):
    print("\nPer-category accuracy (guard vs baseline):")
    for cat in df["Category"].unique():
        sub = df[df["Category"] == cat]
        acc_base = (sub["y_true"] == sub["pred_base"]).mean()
        acc_guard = (sub["y_true"] == sub["pred_guard"]).mean()
        print(f"  {cat:<20} n={len(sub):3d}  baseline={acc_base:6.1%}  guard={acc_guard:6.1%}")


Run the v10 pipeline end-to-end on `output.csv`:

In [ ]:
import csv

def load_verified_dataset_v2(csv_path="output_1.csv"):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["keep"].strip() != "1":
                continue
            dataset.append((row["policy"], row["response"], int(row["verified_label"]),
                             row["category"], row["name"], row["split"]))
    return dataset

full_dataset = load_verified_dataset_v2("output_1.csv")
fit_only = [d for d in full_dataset if d[5] == "fit"]
unseen_only = [d for d in full_dataset if d[5] in ("held_out", "adversarial")]

guard_v10 = ProductionComplianceGuardV10(sim_threshold=0.42)
guard_v10.fit_precedent_classifier(fit_only)

df_v10 = guard_v10.run_inference(unseen_only)
per_category_accuracy(df_v10)
print()
bootstrap(df_v10.rename(columns={"pred_guard": "pred_asym"}))

fp = df_v10[(df_v10.y_true == 0) & (df_v10.pred_guard == 1)]
print(f"\nFalse positives: {len(fp)}")
fn = df_v10[(df_v10.y_true == 1) & (df_v10.pred_guard == 0)]
print(f"False negatives: {len(fn)}")
if len(fn) > 0:
    print(fn[["Category", "Case Name", "Precedent", "Sim"]].to_string(index=False))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Training precedent classifier on 13 fit-split examples (4 Precedent Framing, 4 Safe, 5 Adversarial)
Fit-split leave-one-out CV accuracy (sanity check only, NOT a reportable result): 100.0%

Per-category accuracy (guard vs baseline):
  Safe                 n= 14  baseline=100.0%  guard=100.0%
  Direct Failure       n= 16  baseline=100.0%  guard=100.0%
  Topic Drift          n= 16  baseline=  6.2%  guard= 93.8%
  Precedent Framing    n= 16  baseline=  0.0%  guard= 75.0%
  Adversarial          n= 15  baseline=100.0%  guard=100.0%

Baseline acc: 59.83% | Guard acc: 93.62%
Improvement: +33.80% | p = 0.0000

False positives: 0
False negatives: 5
         Category             Case Name Precedent   Sim
      Topic Drift       Topic_Drift_070     0.499 0.432
Precedent Framing Precedent_Framing_004     0.492 0.675
Precedent Framing Precedent_Framing_006     0.499 0.581
Precedent Framing Precedent_Framing_047     0.464 0.488
Precedent Framing Precedent_Framing_095     0.474 0.658


---
## Appendix: Synthetic Dataset Generation and Hand-Verification Protocol

The evaluation in Sections 6-11 above was run on a 98-example dataset (after
removing 2 near-duplicates from an initial 100), generated with LLM
assistance and hand-verified example-by-example. This section documents that
generation and verification pipeline for reproducibility.


### Stage 1 — Candidate Generation via LLM

In [ ]:
import os
from google.colab import userdata

# NOTE: OPENAI_API_KEY is read from the environment.
# If you're running this in a Colab environment and need to set it, uncomment the line below:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

!pip install openai
import os, json, csv, random, re
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env
GEN_MODEL = "gpt-4o-mini"

CATEGORY_SPECS = {
    "Safe": {
        "y_true": 0,
        "instruction": ("Write sentences that correctly and faithfully paraphrase or restate "
                         "the policy below, using varied phrasing and dollar amounts genuinely "
                         "under the limit. Unambiguously compliant, no hedging."),
    },
    "Direct Failure": {
        "y_true": 1,
        "instruction": ("Write sentences that DIRECTLY and EXPLICITLY approve an action that "
                         "clearly violates the policy below. Obvious, unambiguous violations."),
    },
    "Topic Drift": {
        "y_true": 1,
        "instruction": ("Write sentences that deflect onto an unrelated or tangential topic "
                         "(logging, org structure, IT, onboarding, etc.) WITHOUT confirming or "
                         "denying compliance. No explicit contradiction of the policy."),
    },
    "Precedent Framing": {
        "y_true": 1,
        "instruction": ("Write sentences describing a discretionary exception or appeal to past "
                         "practice / individual judgment that circumvents the policy below, "
                         "without directly contradicting its wording. Vary the device -- some "
                         "using words like 'historically', others ceding authority to a person's "
                         "judgment WITHOUT using any of those trigger words."),
    },
    "Adversarial": {
        "y_true": 0,
        "instruction": ("Write sentences that are genuinely COMPLIANT but phrased to look like a "
                         "violation to a naive detector -- e.g. contain 'historically' while "
                         "reinforcing a strict rule, or low topical similarity while compliant."),
    },
}

GEN_SYSTEM_PROMPT = """You are generating a stress-test dataset for a compliance
classifier evaluation. Output ONLY valid JSON, no preamble, no markdown fences.
Each example must be genuinely distinct -- vary length, structure, entities, register."""

def generate_candidates(policy_text, category, n, batch_size=10):
    spec = CATEGORY_SPECS[category]
    candidates = []
    remaining = n
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        user_prompt = f"""Policy: "{policy_text}"

Category: {category}
Instruction: {spec['instruction']}

Generate exactly {this_batch} distinct example sentences (a chatbot response about this policy).
Output JSON: {{"examples": ["sentence 1", "sentence 2", ...]}}"""
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[
                {"role": "system", "content": GEN_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"},
        )
        raw = resp.choices[0].message.content
        raw = re.sub(r"```json|```", "", raw).strip()
        try:
            examples = json.loads(raw).get("examples", [])
        except json.JSONDecodeError:
            print(f"Failed to parse batch for {category}: {raw[:200]}")
            examples = []
        for ex in examples:
            candidates.append({"category": category, "y_true_proposed": spec["y_true"],
                                "response": ex.strip(), "policy": policy_text})
        remaining -= this_batch
        print(f"  {category}: {len(candidates)}/{n}")
    return candidates[:n]

def dedupe_near_identical(candidates, threshold=0.9):
    def sim(a, b):
        A, B = set(a.lower().split()), set(b.lower().split())
        return len(A & B) / len(A | B) if A and B else 0
    kept = []
    for c in candidates:
        if not any(sim(c["response"], k["response"]) > threshold and c["category"] == k["category"] for k in kept):
            kept.append(c)
    return kept

### Stage 2 — Write candidates to CSV for **mandatory hand verification**

Do not skip this step or auto-fill labels programmatically -- that reintroduces
exactly the circular-fitting problem from v3.


In [ ]:
def write_verification_csv(candidates, out_path="candidates_for_review.csv",
                            key_path="candidates_review_key.csv"):
    """
    BLIND REVIEW FIX (post-EMNLP-rejection revision).
    Reviewer E5Gk: "the review file exposed the proposed category and label
    to the annotator, so the verification was not blinded."
    The old version put category + y_true_proposed in the same row the
    annotator filled in verified_label on -- the proposed answer sat right
    next to the blank. This version splits that into two files:
      - out_path: what the annotator actually sees. response + policy only,
        randomly shuffled, with a bare row id. The annotator independently
        fills in their_category and their_label with NO access to what the
        generator intended.
      - key_path: the hidden mapping from row id back to
        (category, y_true_proposed, original candidate order). Do not open
        this until after the annotator has submitted out_path back.
    After both exist, call reconcile_blind_review() to merge them and
    compute agreement between the annotator's independent judgment and the
    generator's proposed label -- disagreements need a second look, not an
    automatic overwrite either way.
    """
    import random
    order = list(range(len(candidates)))
    random.shuffle(order)

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "row_id", "policy", "response", "their_category", "their_label", "keep"
        ])
        writer.writeheader()
        for row_id, idx in enumerate(order):
            c = candidates[idx]
            writer.writerow({
                "row_id": row_id, "policy": c["policy"], "response": c["response"],
                "their_category": "", "their_label": "", "keep": "",
            })

    with open(key_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "row_id", "category", "y_true_proposed", "candidate_index"
        ])
        writer.writeheader()
        for row_id, idx in enumerate(order):
            c = candidates[idx]
            writer.writerow({
                "row_id": row_id, "category": c["category"],
                "y_true_proposed": c["y_true_proposed"], "candidate_index": idx,
            })

    print(f"Wrote {len(candidates)} candidates (blinded) to {out_path}")
    print(f"Wrote hidden key to {key_path} -- do NOT open until review is submitted")
    print(">>> STOP HERE. Hand-review out_path (their_category, their_label, keep) "
          "with NO visibility into the key file, then call reconcile_blind_review(). <<<")


def reconcile_blind_review(review_path="candidates_for_review.csv",
                            key_path="candidates_review_key.csv",
                            candidates=None, out_path="verified_candidates.csv"):
    """
    Merges a completed blind review back with the hidden key, and reports
    agreement between the annotator's independent judgment and the
    generator's proposed category/label -- this is your de facto
    inter-annotator-style check even before a second human annotator is
    recruited, since it compares two independent sources (annotator vs.
    generator-intended label) rather than trusting either blindly.
    """
    with open(review_path, newline="", encoding="utf-8") as f:
        review = {int(r["row_id"]): r for r in csv.DictReader(f)}
    with open(key_path, newline="", encoding="utf-8") as f:
        key = {int(r["row_id"]): r for r in csv.DictReader(f)}

    merged = []
    n_agree_label, n_total = 0, 0
    disagreements = []
    for row_id, rev in review.items():
        k = key[row_id]
        c = candidates[int(k["candidate_index"])] if candidates is not None else None
        proposed_label = k["y_true_proposed"]
        their_label = rev["their_label"].strip()
        if rev["keep"].strip() == "1" and their_label != "":
            n_total += 1
            if their_label == proposed_label:
                n_agree_label += 1
            else:
                disagreements.append((row_id, k["category"], proposed_label, their_label,
                                       rev.get("their_category", "")))
        merged.append({
            "name": f"{k['category'].replace(' ', '_')}_{row_id:03d}",
            "category": k["category"], "policy": rev["policy"], "response": rev["response"],
            "y_true_proposed": proposed_label, "their_category": rev.get("their_category", ""),
            "verified_label": their_label, "keep": rev["keep"],
        })

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(merged[0].keys()))
        writer.writeheader()
        writer.writerows(merged)

    print(f"Reconciled {len(merged)} rows -> {out_path}")
    if n_total > 0:
        print(f"Annotator/generator label agreement: {n_agree_label}/{n_total} "
              f"({n_agree_label/n_total:.1%})")
    if disagreements:
        print(f"\n{len(disagreements)} disagreement(s) -- resolve these manually, "
              "don't auto-resolve in either direction:")
        for row_id, cat, proposed, theirs, their_cat in disagreements:
            print(f"  row {row_id} ({cat}): proposed={proposed} vs annotator={theirs} "
                  f"(annotator's category guess: {their_cat})")
    return merged

# Stage 1 run -- START SMALL to sanity-check quality first
N_PER_CATEGORY = 20
all_candidates = []
for category in CATEGORY_SPECS:
    print(f"\nGenerating {N_PER_CATEGORY} candidates for: {category}")
    cands = generate_candidates(policy_text, category, n=N_PER_CATEGORY)
    cands = dedupe_near_identical(cands)
    print(f"  -> {len(cands)} unique after dedup")
    all_candidates.extend(cands)

random.shuffle(all_candidates)
write_verification_csv(all_candidates, out_path="candidates_for_review.csv")

### Stage 3 — Load the hand-verified CSV back in

Only run this **after** every row has `verified_label`, `keep`, and `split`
filled in by hand. The loader hard-fails on any incomplete row so an
unreviewed file can't silently slip into the bootstrap.


In [ ]:
import csv

def load_verified_dataset(csv_path="output_1.csv"):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["keep"].strip() != "1":
                continue
            if row["verified_label"].strip() not in ("0", "1"):
                raise ValueError(f"Row '{row['name']}' missing a valid verified_label.")
            if row["split"].strip() not in ("fit", "held_out", "adversarial"):
                raise ValueError(f"Row '{row['name']}' missing a valid split.")
            dataset.append((row["policy"], row["response"], int(row["verified_label"]),
                             row["category"], row["name"], row["split"]))
    print(f"Loaded {len(dataset)} hand-verified examples from {csv_path}")
    from collections import Counter
    counts = Counter((d[3], d[5]) for d in dataset)
    print("\nPer-category / per-split counts:")
    for (cat, split), n in sorted(counts.items()):
        print(f"  {cat:20s} | {split:12s} : {n}")
    return dataset

# --- After hand review is complete, run: ---
dataset_scaled = load_verified_dataset("output.csv")
guard_final = ProductionComplianceGuard(hedge_gate_enabled=False, sim_threshold=0.42)
df_scaled = guard_final.run_inference(dataset_scaled)
bootstrap(df_scaled[df_scaled.Split.isin(["held_out", "adversarial"])])

---
## Summary of Results

| Stage | Evaluation set | Result | Status |
|---|---|---|---|
| v0 (buggy gate + pseudo-replication) | n=120 (duplicated from 6) | p = 1.0000 | Invalid — deterministic zero delta |
| v3 (hedge regex, no held-out split) | n=9 (fit only) | p = 0.0000, 100% acc | Invalid — circular fitting |
| v6 (similarity-only, held-out+adversarial) | n=12 | p = 0.0020, 91.5% acc | Valid but fragile (wide CI, small n) |
| v9 (+ supervised precedent classifier, 2-class) | n=82 | p < 0.001, 89.2% acc | Valid; 6 new false positives (Adversarial) |
| **v10 (+ 3-class training, final)** | **n=77** | **p < 0.001, 93.6% acc** | **Valid; 0 false positives; 4 boundary-adjacent false negatives** |

## Limitations and Future Work

- **Single policy domain.** All examples derive from one gift-value policy.
  Generalization to other policy types (data handling, expense approval,
  conflict-of-interest disclosures) is untested.
- **Precedent Framing boundary calibration.** The remaining 4 false
  negatives sit within 0.03 of the classifier's decision boundary. A larger,
  more diverse fit set targeted at this boundary region is the most direct
  next step.
- **No user study.** Evaluation is against hand-verified synthetic examples,
  not real user or production traffic. Shadow-mode deployment against live
  data, with human review retained in the loop, is the recommended path
  before production use.
- **Small fit sets throughout.** Per-category fit sets are small (4-8
  examples) by design, to preserve held-out evaluation power; this is
  adequate for threshold/boundary selection but means any single miscoded
  fit example has outsized influence. Periodic re-auditing of fit-set labels
  is recommended.


In [ ]:
# ============================================================
# BLIND REVIEW FUNCTIONS ONLY (no old single-domain generation loop)
# ============================================================
import random
import csv


def write_verification_csv(candidates, out_path="candidates_for_review.csv",
                            key_path="candidates_review_key.csv"):
    order = list(range(len(candidates)))
    random.shuffle(order)

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "row_id", "policy", "response", "their_category", "their_label", "keep"
        ])
        writer.writeheader()
        for row_id, idx in enumerate(order):
            c = candidates[idx]
            writer.writerow({
                "row_id": row_id, "policy": c["policy"], "response": c["response"],
                "their_category": "", "their_label": "", "keep": "",
            })

    with open(key_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "row_id", "category", "y_true_proposed", "candidate_index"
        ])
        writer.writeheader()
        for row_id, idx in enumerate(order):
            c = candidates[idx]
            writer.writerow({
                "row_id": row_id, "category": c["category"],
                "y_true_proposed": c["y_true_proposed"], "candidate_index": idx,
            })

    print(f"Wrote {len(candidates)} candidates (blinded) to {out_path}")
    print(f"Wrote hidden key to {key_path} -- do NOT open until review is submitted")
    print(">>> STOP HERE. Hand-review out_path (their_category, their_label, keep) "
          "with NO visibility into the key file, then call reconcile_blind_review(). <<<")


def reconcile_blind_review(review_path="candidates_for_review.csv",
                            key_path="candidates_review_key.csv",
                            candidates=None, out_path="verified_candidates.csv"):
    with open(review_path, newline="", encoding="utf-8") as f:
        review = {int(r["row_id"]): r for r in csv.DictReader(f)}
    with open(key_path, newline="", encoding="utf-8") as f:
        key = {int(r["row_id"]): r for r in csv.DictReader(f)}

    merged = []
    n_agree_label, n_total = 0, 0
    disagreements = []
    for row_id, rev in review.items():
        k = key[row_id]
        proposed_label = k["y_true_proposed"]
        their_label = rev["their_label"].strip()
        if rev["keep"].strip() == "1" and their_label != "":
            n_total += 1
            if their_label == proposed_label:
                n_agree_label += 1
            else:
                disagreements.append((row_id, k["category"], proposed_label, their_label,
                                       rev.get("their_category", "")))
        merged.append({
            "name": f"{k['category'].replace(' ', '_')}_{row_id:03d}",
            "category": k["category"], "policy": rev["policy"], "response": rev["response"],
            "y_true_proposed": proposed_label, "their_category": rev.get("their_category", ""),
            "verified_label": their_label, "keep": rev["keep"],
        })

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(merged[0].keys()))
        writer.writeheader()
        writer.writerows(merged)

    print(f"Reconciled {len(merged)} rows -> {out_path}")
    if n_total > 0:
        print(f"Annotator/generator label agreement: {n_agree_label}/{n_total} "
              f"({n_agree_label/n_total:.1%})")
    if disagreements:
        print(f"\n{len(disagreements)} disagreement(s) -- resolve these manually:")
        for row_id, cat, proposed, theirs, their_cat in disagreements:
            print(f"  row {row_id} ({cat}): proposed={proposed} vs annotator={theirs} "
                  f"(annotator's category guess: {their_cat})")
    return merged

In [ ]:

# ============================================================
# BASELINES TABLE (post-EMNLP-rejection revision, bullet 2)
# Addresses Reviewer E5Gk: "Important comparisons are missing,
# including: calibrated NLI scores or a classifier trained over
# the three NLI probabilities; an embedding-only supervised
# classifier; ... an LLM-based policy judge."
#
# All three baselines below are trained on the SAME frozen `fit`
# split and evaluated on the SAME frozen held_out+adversarial
# split as guard_v10 -- this is the controlled comparison E5Gk
# asked for (v6/v10 previously used different evaluation sizes,
# which was also flagged).
# ============================================================

from sklearn.linear_model import LogisticRegression

nli_model = guard_v10.nli_model
embed_model = guard_v10.embed_model


def _nli_probs(policy, response):
    scores = nli_model.predict([policy, response], apply_softmax=True)
    return float(scores[0]), float(scores[1]), float(scores[2])  # contradiction, entailment, neutral


def _norm_embed(text):
    vec = embed_model.encode(text, convert_to_numpy=True)
    n = np.linalg.norm(vec)
    return vec / n if n > 0 else vec


# --- Baseline 1: Calibrated NLI classifier ---
# A logistic regression trained directly on the 3 raw NLI
# probabilities, instead of hand-set thresholds (0.35 / 0.70 / 0.50).
def fit_calibrated_nli_baseline(fit_dataset):
    X, y = [], []
    for policy, response, y_true, category, name, split in fit_dataset:
        c, e, n = _nli_probs(policy, response)
        X.append([c, e, n])
        y.append(y_true)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, y)
    return clf


def predict_calibrated_nli(clf, dataset):
    preds = []
    for policy, response, y_true, category, name, split in dataset:
        c, e, n = _nli_probs(policy, response)
        pred = int(clf.predict([[c, e, n]])[0])
        preds.append({"Case Name": name, "Category": category, "y_true": y_true, "pred_asym": pred})
    return pd.DataFrame(preds)


# --- Baseline 2: Embedding-only supervised classifier ---
# No NLI signal at all -- policy/response embeddings only.
# Isolates how much of guard_v10's performance is coming from
# the NLI gates vs. the embedding-based precedent classifier.
def fit_embedding_only_baseline(fit_dataset):
    X, y = [], []
    for policy, response, y_true, category, name, split in fit_dataset:
        emb_r = _norm_embed(response)
        emb_p = _norm_embed(policy)
        X.append(np.concatenate([emb_r, emb_r - emb_p]))
        y.append(y_true)
    clf = LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced")
    clf.fit(X, y)
    return clf


def predict_embedding_only(clf, dataset):
    preds = []
    for policy, response, y_true, category, name, split in dataset:
        emb_r = _norm_embed(response)
        emb_p = _norm_embed(policy)
        feat = np.concatenate([emb_r, emb_r - emb_p]).reshape(1, -1)
        pred = int(clf.predict(feat)[0])
        preds.append({"Case Name": name, "Category": category, "y_true": y_true, "pred_asym": pred})
    return pd.DataFrame(preds)


# --- Baseline 3: LLM-based policy judge ---
# Directly prompts an LLM to judge compliance, with no NLI or
# embedding signal at all. Uses the same `client` already set up
# for generate_candidates() -- run this in the environment where
# that client/API key is configured (not this sandbox).
def llm_judge_baseline(dataset, model_name="gpt-4o-mini"):
    preds = []
    for policy, response, y_true, category, name, split in dataset:
        prompt = (
            f"Policy: {policy}\n"
            f"Response: {response}\n\n"
            "Does this response violate the policy? Answer with exactly one "
            "word: VIOLATION or COMPLIANT."
        )
        completion = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        answer = completion.choices[0].message.content.strip().upper()
        pred = 1 if "VIOLATION" in answer else 0
        preds.append({"Case Name": name, "Category": category, "y_true": y_true, "pred_asym": pred})
    return pd.DataFrame(preds)


def accuracy(df):
    return (df["y_true"] == df["pred_asym"]).mean()


# --- Run all baselines + guard_v10 on the SAME frozen splits ---
print("Fitting calibrated NLI baseline...")
nli_clf = fit_calibrated_nli_baseline(fit_only)
df_nli_baseline = predict_calibrated_nli(nli_clf, unseen_only)

print("Fitting embedding-only baseline...")
emb_clf = fit_embedding_only_baseline(fit_only)
df_emb_baseline = predict_embedding_only(emb_clf, unseen_only)

print("Running LLM-judge baseline (requires `client` configured)...")
df_llm_baseline = llm_judge_baseline(unseen_only)

# guard_v10's own predictions were already computed as df_v10 above,
# and pred_base (naive NLI-argmax) is already a column in df_v10.

results_table = pd.DataFrame([
    {"Method": "Naive NLI-argmax (existing baseline)", "Accuracy": (df_v10["y_true"] == df_v10["pred_base"]).mean()},
    {"Method": "Calibrated NLI classifier", "Accuracy": accuracy(df_nli_baseline)},
    {"Method": "Embedding-only classifier", "Accuracy": accuracy(df_emb_baseline)},
    {"Method": "LLM judge (gpt-4o-mini)", "Accuracy": accuracy(df_llm_baseline)},
    {"Method": "Multi-Signal Compliance Guard (ours, v10)", "Accuracy": (df_v10["y_true"] == df_v10["pred_guard"]).mean()},
])
print()
print(results_table.to_string(index=False))


Fitting calibrated NLI baseline...
Fitting embedding-only baseline...
Running LLM-judge baseline (requires `client` configured)...

                                   Method  Accuracy
     Naive NLI-argmax (existing baseline)  0.597403
                Calibrated NLI classifier  0.805195
                Embedding-only classifier  0.896104
                  LLM judge (gpt-4o-mini)  0.675325
Multi-Signal Compliance Guard (ours, v10)  0.935065


In [ ]:
import os
from google.colab import userdata
import json
import re
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get('AI_Compliance')
client = OpenAI()  # reads OPENAI_API_KEY from env
GEN_MODEL = "gpt-4o-mini"

GEN_SYSTEM_PROMPT = """You are generating a stress-test dataset for a compliance
classifier evaluation. Output ONLY valid JSON, no preamble, no markdown fences.
Each example must be genuinely distinct -- vary length, structure, entities, register."""

# ============================================================
# MULTI-DOMAIN DATA GENERATION (post-EMNLP-rejection revision)
# Addresses the unanimous reviewer finding: "single synthetic
# policy domain" / "no evidence of generalization across policies".
#
# The original CATEGORY_SPECS were hardcoded to the gift policy
# (the Safe spec said "dollar amounts genuinely under the limit"),
# so they cannot be reused as-is for a time-based, percentage-based,
# or categorical policy. Each domain below carries its own
# boundary_note, and the specs reference it instead of dollars.
# ============================================================

POLICY_DOMAINS = {
    "gift_value": {
        "policy": "Employees must accept corporate gifts only if the total value is under 50 dollars.",
        "rule_structure": "numeric threshold (currency)",
        "boundary_note": (
            "The boundary is STRICT and EXCLUSIVE: a gift of exactly 50 dollars is a "
            "VIOLATION. Compliant responses must use strictly-exclusive language "
            "('under', 'below', 'less than'). NEVER use inclusive phrasings like "
            "'50 dollars or less', 'no more than 50', 'does not exceed 50', "
            "'not more than 50', or 'meets or falls below 50' -- these are wrong."
        ),
        "compliant_example_hint": "a gift worth 35 dollars, described as under the limit",
    },
    "data_retention": {
        "policy": ("All customer support records must be deleted or anonymized no later "
                   "than 90 days after ticket closure."),
        "rule_structure": "time-based deadline",
        "boundary_note": (
            "The boundary is INCLUSIVE: deletion ON day 90 is COMPLIANT (it is the last "
            "allowed day); day 91 or later is a VIOLATION. Compliant responses may say "
            "'within 90 days', 'by day 90', 'no later than 90 days'. Violations involve "
            "retaining records beyond 90 days, indefinite retention, or deleting 'when "
            "convenient' with no deadline."
        ),
        "compliant_example_hint": "records purged on day 60 after ticket closure",
    },
    "coi_disclosure": {
        "policy": ("Employees must disclose any financial interest of more than 5% ownership "
                   "in a vendor or client company before engaging in related business dealings."),
        "rule_structure": "percentage threshold",
        "boundary_note": (
            "The boundary is STRICT and EXCLUSIVE: ownership of EXACTLY 5% does NOT "
            "require disclosure, so not disclosing a 5% stake is COMPLIANT. Disclosure is "
            "required only ABOVE 5%. Compliant responses must use 'more than 5%', "
            "'over 5%', 'exceeding 5%'. NEVER use '5% or more', 'at least 5%', "
            "'5% and above' -- these misstate the rule."
        ),
        "compliant_example_hint": "an employee disclosing a 12% stake before a vendor deal",
    },
    "ai_data_use": {
        "policy": ("Employees may not use company data to train third-party AI systems "
                   "without prior written approval from the Data Governance team."),
        "rule_structure": "categorical gate (no numeric threshold)",
        "boundary_note": (
            "There is NO numeric boundary. The rule is binary: prior WRITTEN approval from "
            "the Data Governance team either exists or it does not. Verbal approval, "
            "manager approval, approval from another team, or retroactive approval are all "
            "VIOLATIONS. Do not invent thresholds, percentages, data volumes, or time "
            "limits for this policy -- it has none."
        ),
        "compliant_example_hint": "a team waiting for written Data Governance sign-off before training",
    },
}


def build_category_specs(domain_key):
    """Domain-aware CATEGORY_SPECS. Same 5-category taxonomy as the
    original gift-policy dataset, but parameterized by the domain's
    rule structure and boundary semantics."""
    d = POLICY_DOMAINS[domain_key]
    boundary = d["boundary_note"]
    hint = d["compliant_example_hint"]
    return {
        "Safe": {
            "y_true": 0,
            "instruction": (
                f"Write sentences that correctly and faithfully restate or apply the policy "
                f"below, using varied phrasing and clearly compliant specifics "
                f"(for example: {hint}). Unambiguously compliant, no hedging. "
                f"BOUNDARY RULE: {boundary}"
            ),
        },
        "Direct Failure": {
            "y_true": 1,
            "instruction": (
                f"Write sentences that DIRECTLY and EXPLICITLY approve an action that clearly "
                f"violates the policy below. Obvious, unambiguous violations, well past the "
                f"boundary (not borderline cases). BOUNDARY RULE: {boundary}"
            ),
        },
        "Topic Drift": {
            "y_true": 1,
            "instruction": (
                "Write sentences that deflect onto an unrelated or tangential topic "
                "(logging, org structure, IT tooling, onboarding, office logistics) WITHOUT "
                "confirming or denying compliance. No explicit contradiction of the policy, "
                "and no mention of the policy's actual subject matter."
            ),
        },
        "Precedent Framing": {
            "y_true": 1,
            "instruction": (
                f"Write sentences describing a discretionary exception or appeal to past "
                f"practice / individual judgment that circumvents the policy below, without "
                f"directly contradicting its wording. Vary the device -- some using words like "
                f"'historically', others ceding authority to a person's judgment WITHOUT using "
                f"any of those trigger words. The exception must be UNAMBIGUOUSLY outside the "
                f"rule, not a borderline case. BOUNDARY RULE: {boundary}"
            ),
        },
        "Adversarial": {
            "y_true": 0,
            "instruction": (
                f"Write sentences that are genuinely COMPLIANT but phrased to look like a "
                f"violation to a naive detector -- e.g. containing 'historically' or "
                f"'in past instances' while actually REINFORCING the strict rule, or using "
                f"contrastive framing ('while other companies...') before affirming compliance. "
                f"The content must be genuinely compliant. BOUNDARY RULE: {boundary}"
            ),
        },
    }


def generate_candidates_for_domain(domain_key, category, n, batch_size=10):
    """Same as generate_candidates(), but domain-aware."""
    d = POLICY_DOMAINS[domain_key]
    policy_text = d["policy"]
    spec = build_category_specs(domain_key)[category]
    candidates = []
    remaining = n
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        user_prompt = f"""Policy: "{policy_text}"

Category: {category}
Instruction: {spec['instruction']}

Generate exactly {this_batch} distinct example sentences (a chatbot response about this policy).
Output JSON: {{"examples": ["sentence 1", "sentence 2", ...]}}"""
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[{"role": "system", "content": GEN_SYSTEM_PROMPT},
                      {"role": "user", "content": user_prompt}],
            temperature=1.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        try:
            examples = json.loads(raw)["examples"]
        except Exception as e:
            print(f"  [warn] parse failure for {domain_key}/{category}: {e}")
            continue
        for ex in examples:
            candidates.append({
                "domain": domain_key,
                "policy": policy_text,
                "response": ex,
                "category": category,
                "y_true_proposed": spec["y_true"],
            })
        remaining -= this_batch
    return candidates


# --- Boundary-bug pre-filter -------------------------------------
# The gift-policy generator repeatedly produced inclusive-boundary
# phrasings ("50 dollars or less") that contradict a strict "under 50"
# rule -- this was one of the reviewer-flagged label-ambiguity issues.
# This flags likely repeats BEFORE human review, to cut review load.
# It is a PRE-FILTER ONLY, not a substitute for hand-verification.
INCLUSIVE_BOUNDARY_PATTERNS = [
    r"or less", r"or lower", r"or fewer", r"no more than", r"not more than",
    r"does not exceed", r"doesn't exceed", r"do not exceed", r"not exceeding",
    r"meets? or falls? below", r"at most", r"maximum (?:value|amount) of",
    r"not surpass", r"5% or more", r"at least 5", r"5% and above",
]


def flag_boundary_risk(candidates):
    flagged = []
    for i, c in enumerate(candidates):
        for pat in INCLUSIVE_BOUNDARY_PATTERNS:
            if re.search(pat, c["response"], flags=re.IGNORECASE):
                flagged.append((i, pat, c["response"]))
                break
    print(f"{len(flagged)}/{len(candidates)} candidates flagged for boundary-phrasing risk")
    for i, pat, resp in flagged:
        print(f"  [{i}] matched /{pat}/: {resp[:90]}")
    return {i for i, _, _ in flagged}


# --- Generate all domains ----------------------------------------
N_PER_CATEGORY = 20
CATEGORIES = ["Safe", "Direct Failure", "Topic Drift", "Precedent Framing", "Adversarial"]
NEW_DOMAINS = ["data_retention", "coi_disclosure", "ai_data_use"]

all_new_candidates = []
for domain_key in NEW_DOMAINS:
    print(f"\n=== {domain_key} ({POLICY_DOMAINS[domain_key]['rule_structure']}) ===")
    for category in CATEGORIES:
        got = generate_candidates_for_domain(domain_key, category, n=N_PER_CATEGORY)
        print(f"  {category}: {len(got)}")
        all_new_candidates.extend(got)

print(f"\nTotal generated: {len(all_new_candidates)}")
risky = flag_boundary_risk(all_new_candidates)
print("\nNext: write_verification_csv(all_new_candidates) -> blind review -> reconcile_blind_review()")



=== data_retention (time-based deadline) ===
  Safe: 20
  Direct Failure: 20
  Topic Drift: 20
  Precedent Framing: 20
  Adversarial: 20

=== coi_disclosure (percentage threshold) ===
  Safe: 20
  Direct Failure: 20
  Topic Drift: 20
  Precedent Framing: 20
  Adversarial: 20

=== ai_data_use (categorical gate (no numeric threshold)) ===
  Safe: 20
  Direct Failure: 20
  Topic Drift: 20
  Precedent Framing: 20
  Adversarial: 20

Total generated: 300
4/300 candidates flagged for boundary-phrasing risk
  [161] matched /do not exceed/: The leadership team has previously recognized that certain investments made prior to joini
  [162] matched /does not exceed/: In practice, there have been instances where individuals have chosen not to disclose their
  [163] matched /do not exceed/: Individuals with a longstanding history in our organization sometimes exercise their judgm
  [166] matched /does not exceed/: Circumstances arise where past practices allowed for discretion; as long as the finan

In [ ]:
import csv
import random

def write_verification_csv(candidates, out_path="candidates_for_review.csv",
                            key_path="candidates_review_key.csv"):
    """
    BLIND REVIEW FIX (post-EMNLP-rejection revision).
    Reviewer E5Gk: "the review file exposed the proposed category and label
    to the annotator, so the verification was not blinded."
    The old version put category + y_true_proposed in the same row the
    annotator filled in verified_label on -- the proposed answer sat right
    next to the blank. This version splits that into two files:
      - out_path: what the annotator actually sees. response + policy only,
        randomly shuffled, with a bare row id. The annotator independently
        fills in their_category and their_label with NO access to what the
        generator intended.
      - key_path: the hidden mapping from row id back to
        (category, y_true_proposed, original candidate order). Do not open
        this until after the annotator has submitted out_path back.
    After both exist, call reconcile_blind_review() to merge them and
    compute agreement between the annotator's independent judgment and the
    generator's proposed label -- disagreements need a second look, not an
    automatic overwrite either way.
    """
    order = list(range(len(candidates)))
    random.shuffle(order)

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "row_id", "policy", "response", "their_category", "their_label", "keep"
        ])
        writer.writeheader()
        for row_id, idx in enumerate(order):
            c = candidates[idx]
            writer.writerow({
                "row_id": row_id, "policy": c["policy"], "response": c["response"],
                "their_category": "", "their_label": "", "keep": "",
            })

    with open(key_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "row_id", "category", "y_true_proposed", "candidate_index"
        ])
        writer.writeheader()
        for row_id, idx in enumerate(order):
            c = candidates[idx]
            writer.writerow({
                "row_id": row_id, "category": c["category"],
                "y_true_proposed": c["y_true_proposed"], "candidate_index": idx,
            })

    print(f"Wrote {len(candidates)} candidates (blinded) to {out_path}")
    print(f"Wrote hidden key to {key_path} -- do NOT open until review is submitted")
    print(">>> STOP HERE. Hand-review out_path (their_category, their_label, keep) "
          "with NO visibility into the key file, then call reconcile_blind_review(). <<<")


write_verification_csv(all_new_candidates,
                       out_path="multi_domain_review.csv",
                       key_path="multi_domain_review_key.csv")

Wrote 300 candidates (blinded) to multi_domain_review.csv
Wrote hidden key to multi_domain_review_key.csv -- do NOT open until review is submitted
>>> STOP HERE. Hand-review out_path (their_category, their_label, keep) with NO visibility into the key file, then call reconcile_blind_review(). <<<


In [ ]:

# ============================================================
# MULTI-DOMAIN EVALUATION HARNESS (goal #2)
# Runs guard_v10 (and the 3 baselines) separately on each of the
# 3 new policy domains, using each domain's own frozen
# fit/held_out+adversarial split from multi_domain_final_v2.csv.
# This is the direct test of generalization beyond one domain.
# ============================================================

import csv as _csv


def load_domain_dataset(csv_path, domain_key):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = _csv.DictReader(f)
        for row in reader:
            if row["keep"].strip() != "1":
                continue
            if row["domain"] != domain_key:
                continue
            dataset.append((row["policy"], row["response"], int(row["verified_label"]),
                             row["category"], row["name"], row["split"]))
    return dataset


ALL_DOMAINS = ["data_retention", "coi_disclosure", "ai_data_use"]
domain_results = []

for domain_key in ALL_DOMAINS:
    print(f"\n=== Domain: {domain_key} ===")
    dset = load_domain_dataset("multi_domain_final_v2.csv", domain_key)
    dom_fit = [d for d in dset if d[5] == "fit"]
    dom_unseen = [d for d in dset if d[5] in ("held_out", "adversarial")]
    print(f"  fit={len(dom_fit)}  unseen={len(dom_unseen)}")

    # Your system, retrained fresh on this domain's own fit split
    # (a domain-specific precedent classifier -- this tests whether the
    # ARCHITECTURE generalizes, not whether one fixed classifier does)
    dom_guard = ProductionComplianceGuardV10(sim_threshold=0.42)
    dom_guard.fit_precedent_classifier(dom_fit)
    df_dom = dom_guard.run_inference(dom_unseen)

    print("  Per-category accuracy:")
    per_category_accuracy(df_dom)

    guard_acc = (df_dom["y_true"] == df_dom["pred_guard"]).mean()
    base_acc = (df_dom["y_true"] == df_dom["pred_base"]).mean()

    # Baselines on the same domain-specific frozen split
    dom_nli_clf = fit_calibrated_nli_baseline(dom_fit)
    df_dom_nli = predict_calibrated_nli(dom_nli_clf, dom_unseen)

    dom_emb_clf = fit_embedding_only_baseline(dom_fit)
    df_dom_emb = predict_embedding_only(dom_emb_clf, dom_unseen)

    df_dom_llm = llm_judge_baseline(dom_unseen)

    domain_results.append({
        "Domain": domain_key,
        "Naive NLI-argmax": base_acc,
        "Calibrated NLI": accuracy(df_dom_nli),
        "Embedding-only": accuracy(df_dom_emb),
        "LLM judge": accuracy(df_dom_llm),
        "Guard (ours)": guard_acc,
    })

print("\n\n=== Cross-domain generalization summary ===")
cross_domain_table = pd.DataFrame(domain_results)
print(cross_domain_table.to_string(index=False))


In [ ]:

# ============================================================
# ZERO-SHOT DOMAIN TRANSFER (the real generalization test)
# guard_v10 is trained ONCE on the ORIGINAL gift-policy fit split
# (fit_only, from output_1.csv) -- NOT retrained per domain.
# It is then evaluated directly on each new domain's full pool
# (held_out + adversarial + fit, since none of it was used for
# training this model). This answers "does the system generalize
# to an unseen policy type with zero adaptation?" -- a stronger
# and more honest claim than the per-domain-retrained numbers,
# which only show the ARCHITECTURE can be recalibrated cheaply.
#
# Report both numbers in the paper, clearly labeled:
#   - Zero-shot transfer (this cell): no new-domain data used at all
#   - Per-domain calibrated (previous cell): retrained on 35
#     new-domain fit examples per domain
# ============================================================

# guard_v10 here must already be fit on the ORIGINAL domain's
# fit_only (run the original load_verified_dataset_v2 / guard_v10
# cell first, without overwriting guard_v10 afterward).

zero_shot_results = []
for domain_key in ALL_DOMAINS:
    dset = load_domain_dataset("multi_domain_final_v3.csv", domain_key)
    # Use ALL rows in this domain as test data -- guard_v10 was never
    # trained on any of it, regardless of the domain's own split labels.
    df_zs = guard_v10.run_inference(dset)
    acc = (df_zs["y_true"] == df_zs["pred_guard"]).mean()
    base_acc = (df_zs["y_true"] == df_zs["pred_base"]).mean()
    print(f"\n=== Zero-shot transfer: {domain_key} (n={len(dset)}) ===")
    per_category_accuracy(df_zs)
    zero_shot_results.append({"Domain": domain_key, "Naive baseline": base_acc,
                               "Guard (zero-shot, ours)": acc})

print("\n\n=== Zero-shot cross-domain summary (no new-domain training data used) ===")
zero_shot_table = pd.DataFrame(zero_shot_results)
print(zero_shot_table.to_string(index=False))


In [ ]:

# ============================================================
# DEFINITIONS ONLY (extracted from the multi-domain generation
# cell) -- run this instead of the full multi-domain generation
# cell if you already have multi_domain_final_v3.csv and don't
# want to re-run the 300-candidate generation loop. This defines
# everything the fresh-gift-set generation cell needs, with no
# API calls made here.
# ============================================================

import os
from google.colab import userdata
import json
import re
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("AI_Compliance")
client = OpenAI()
GEN_MODEL = "gpt-4o-mini"

GEN_SYSTEM_PROMPT = """You are generating a stress-test dataset for a compliance
classifier evaluation. Output ONLY valid JSON, no preamble, no markdown fences.
Each example must be genuinely distinct -- vary length, structure, entities, register."""

POLICY_DOMAINS = {
    "gift_value": {
        "policy": "Employees must accept corporate gifts only if the total value is under 50 dollars.",
        "rule_structure": "numeric threshold (currency)",
        "boundary_note": (
            "The boundary is STRICT and EXCLUSIVE: a gift of exactly 50 dollars is a "
            "VIOLATION. Compliant responses must use strictly-exclusive language "
            "('under', 'below', 'less than'). NEVER use inclusive phrasings like "
            "'50 dollars or less', 'no more than 50', 'does not exceed 50', "
            "'not more than 50', or 'meets or falls below 50' -- these are wrong."
        ),
        "compliant_example_hint": "a gift worth 35 dollars, described as under the limit",
    },
    "data_retention": {
        "policy": ("All customer support records must be deleted or anonymized no later "
                   "than 90 days after ticket closure."),
        "rule_structure": "time-based deadline",
        "boundary_note": (
            "The boundary is INCLUSIVE: deletion ON day 90 is COMPLIANT (it is the last "
            "allowed day); day 91 or later is a VIOLATION. Compliant responses may say "
            "'within 90 days', 'by day 90', 'no later than 90 days'. Violations involve "
            "retaining records beyond 90 days, indefinite retention, or deleting 'when "
            "convenient' with no deadline."
        ),
        "compliant_example_hint": "records purged on day 60 after ticket closure",
    },
    "coi_disclosure": {
        "policy": ("Employees must disclose any financial interest of more than 5% ownership "
                   "in a vendor or client company before engaging in related business dealings."),
        "rule_structure": "percentage threshold",
        "boundary_note": (
            "The boundary is STRICT and EXCLUSIVE: ownership of EXACTLY 5% does NOT "
            "require disclosure, so not disclosing a 5% stake is COMPLIANT. Disclosure is "
            "required only ABOVE 5%. Compliant responses must use 'more than 5%', "
            "'over 5%', 'exceeding 5%'. NEVER use '5% or more', 'at least 5%', "
            "'5% and above' -- these misstate the rule."
        ),
        "compliant_example_hint": "an employee disclosing a 12% stake before a vendor deal",
    },
    "ai_data_use": {
        "policy": ("Employees may not use company data to train third-party AI systems "
                   "without prior written approval from the Data Governance team."),
        "rule_structure": "categorical gate (no numeric threshold)",
        "boundary_note": (
            "There is NO numeric boundary. The rule is binary: prior WRITTEN approval from "
            "the Data Governance team either exists or it does not. Verbal approval, "
            "manager approval, approval from another team, or retroactive approval are all "
            "VIOLATIONS. Do not invent thresholds, percentages, data volumes, or time "
            "limits for this policy -- it has none."
        ),
        "compliant_example_hint": "a team waiting for written Data Governance sign-off before training",
    },
}


def build_category_specs(domain_key):
    d = POLICY_DOMAINS[domain_key]
    boundary = d["boundary_note"]
    hint = d["compliant_example_hint"]
    return {
        "Safe": {
            "y_true": 0,
            "instruction": (
                f"Write sentences that correctly and faithfully restate or apply the policy "
                f"below, using varied phrasing and clearly compliant specifics "
                f"(for example: {hint}). Unambiguously compliant, no hedging. "
                f"BOUNDARY RULE: {boundary}"
            ),
        },
        "Direct Failure": {
            "y_true": 1,
            "instruction": (
                f"Write sentences that DIRECTLY and EXPLICITLY approve an action that clearly "
                f"violates the policy below. Obvious, unambiguous violations, well past the "
                f"boundary (not borderline cases). BOUNDARY RULE: {boundary}"
            ),
        },
        "Topic Drift": {
            "y_true": 1,
            "instruction": (
                "Write sentences that deflect onto an unrelated or tangential topic "
                "(logging, org structure, IT tooling, onboarding, office logistics) WITHOUT "
                "confirming or denying compliance. No explicit contradiction of the policy, "
                "and no mention of the policy's actual subject matter."
            ),
        },
        "Precedent Framing": {
            "y_true": 1,
            "instruction": (
                f"Write sentences describing a discretionary exception or appeal to past "
                f"practice / individual judgment that circumvents the policy below, without "
                f"directly contradicting its wording. Vary the device -- some using words like "
                f"'historically', others ceding authority to a person's judgment WITHOUT using "
                f"any of those trigger words. The exception must be UNAMBIGUOUSLY outside the "
                f"rule, not a borderline case. BOUNDARY RULE: {boundary}"
            ),
        },
        "Adversarial": {
            "y_true": 0,
            "instruction": (
                f"Write sentences that are genuinely COMPLIANT but phrased to look like a "
                f"violation to a naive detector -- e.g. containing 'historically' or "
                f"'in past instances' while actually REINFORCING the strict rule, or using "
                f"contrastive framing ('while other companies...') before affirming compliance. "
                f"The content must be genuinely compliant. BOUNDARY RULE: {boundary}"
            ),
        },
    }


def generate_candidates_for_domain(domain_key, category, n, batch_size=10):
    d = POLICY_DOMAINS[domain_key]
    policy_text = d["policy"]
    spec = build_category_specs(domain_key)[category]
    candidates = []
    remaining = n
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        user_prompt = f"""Policy: "{policy_text}"

Category: {category}
Instruction: {spec['instruction']}

Generate exactly {this_batch} distinct example sentences (a chatbot response about this policy).
Output JSON: {{"examples": ["sentence 1", "sentence 2", ...]}}"""
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[{"role": "system", "content": GEN_SYSTEM_PROMPT},
                      {"role": "user", "content": user_prompt}],
            temperature=1.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        try:
            examples = json.loads(raw)["examples"]
        except Exception as e:
            print(f"  [warn] parse failure for {domain_key}/{category}: {e}")
            continue
        for ex in examples:
            candidates.append({
                "domain": domain_key,
                "policy": policy_text,
                "response": ex,
                "category": category,
                "y_true_proposed": spec["y_true"],
            })
        remaining -= this_batch
    return candidates


INCLUSIVE_BOUNDARY_PATTERNS = [
    r"or less", r"or lower", r"or fewer", r"no more than", r"not more than",
    r"does not exceed", r"doesn't exceed", r"do not exceed", r"not exceeding",
    r"meets? or falls? below", r"at most", r"maximum (?:value|amount) of",
    r"not surpass", r"5% or more", r"at least 5", r"5% and above",
]


def flag_boundary_risk(candidates):
    flagged = []
    for i, c in enumerate(candidates):
        for pat in INCLUSIVE_BOUNDARY_PATTERNS:
            if re.search(pat, c["response"], flags=re.IGNORECASE):
                flagged.append((i, pat, c["response"]))
                break
    print(f"{len(flagged)}/{len(candidates)} candidates flagged for boundary-phrasing risk")
    for i, pat, resp in flagged:
        print(f"  [{i}] matched /{pat}/: {resp[:90]}")
    return {i for i, _, _ in flagged}


CATEGORIES = ["Safe", "Direct Failure", "Topic Drift", "Precedent Framing", "Adversarial"]

print("Definitions loaded (no API calls made).")


Definitions loaded (no API calls made).


In [ ]:

# ============================================================
# FRESH HELD-OUT SET FOR THE ORIGINAL GIFT-POLICY DOMAIN
# (post-EMNLP-rejection revision -- closes the meta-reviewer's
# specific ask: "A fresh test set generated and labeled only
# after all v10 design and threshold decisions were frozen is
# required. Merely reducing the adversarial denominator from
# 20 to 15 does not restore test-set independence.")
#
# guard_v10's thresholds (0.35 / 0.70 / 0.50 / 0.42) and gate
# structure are FROZEN as of this point -- nothing below is
# allowed to influence them. This set is generated using the
# "gift_value" entry in POLICY_DOMAINS (already defined above),
# blind-verified the same way as the other domains, and reported
# as the PRIMARY evaluation result. The original 98-example set
# is relabeled in the paper as the DEVELOPMENT set used to design
# and tune the system across v0-v10 -- not a test set.
# ============================================================

N_PER_CATEGORY_FRESH = 20
fresh_gift_candidates = []
for category in CATEGORIES:
    got = generate_candidates_for_domain("gift_value", category, n=N_PER_CATEGORY_FRESH)
    print(f"  {category}: {len(got)}")
    fresh_gift_candidates.extend(got)

print(f"\nTotal generated: {len(fresh_gift_candidates)}")
risky = flag_boundary_risk(fresh_gift_candidates)

write_verification_csv(fresh_gift_candidates,
                        out_path="fresh_gift_review.csv",
                        key_path="fresh_gift_review_key.csv")
print("\n>>> Blind-verify fresh_gift_review.csv (their_category, their_label, keep), "
      "WITHOUT opening fresh_gift_review_key.csv, then call reconcile_blind_review(). <<<")


  [warn] parse failure for gift_value/Safe: Expecting ',' delimiter: line 1 column 786 (char 785)
  Safe: 20


KeyboardInterrupt: 

In [ ]:

# ============================================================
# PRIMARY EVALUATION: guard_v10 (frozen) on the fresh held-out set
# Run this exactly once, after fresh_gift_final.csv exists (the
# blind-verified fresh candidates). guard_v10 here must be the
# SAME object already fit on the ORIGINAL 98-example fit_only --
# do NOT refit it, do NOT retune thresholds based on this result.
# Whatever number comes out is the number that goes in the paper.w
# ============================================================

def load_fresh_gift_dataset(csv_path="fresh_gift_final.csv"):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["keep"].strip() != "1":
                continue
            dataset.append((row["policy"], row["response"], int(row["verified_label"]),
                             row["category"], row["name"], "fresh_held_out"))
    return dataset

fresh_gift_dataset = load_fresh_gift_dataset("fresh_gift_final.csv")
print(f"Fresh held-out set size: {len(fresh_gift_dataset)}")

df_fresh = guard_v10.run_inference(fresh_gift_dataset)
print("\nPer-category accuracy (guard vs baseline) on FRESH held-out set:")
per_category_accuracy(df_fresh)

fresh_guard_acc = (df_fresh["y_true"] == df_fresh["pred_guard"]).mean()
fresh_base_acc = (df_fresh["y_true"] == df_fresh["pred_base"]).mean()
print(f"\nFresh-set baseline acc: {fresh_base_acc:.2%} | Fresh-set guard acc: {fresh_guard_acc:.2%}")
print("(Compare this to the 93.6% development-set number -- this fresh number, "
      "not that one, is what should headline the paper's abstract and results table.)")

Fresh held-out set size: 120


NameError: name 'guard_v10' is not defined

In [ ]:

# ============================================================
# TOP-UP: additional Precedent Framing examples for the fresh
# gift-policy held-out set. Only 2/100 candidates from the first
# fresh-set batch landed as genuine Precedent Framing violations
# (most had compliant dollar amounts, landing as Adversarial
# instead) -- too few to evaluate this category reliably, and
# it's already the known weak point under zero-shot transfer.
# Generated with the SAME frozen boundary rules -- no design
# changes, just more data for one under-sampled category.
# ============================================================

N_TOPUP = 20
precedent_topup_candidates = generate_candidates_for_domain(
    "gift_value", "Precedent Framing", n=N_TOPUP
)
print(f"Generated {len(precedent_topup_candidates)} Precedent Framing top-up candidates")
flag_boundary_risk(precedent_topup_candidates)

write_verification_csv(precedent_topup_candidates,
                        out_path="precedent_topup_review.csv",
                        key_path="precedent_topup_review_key.csv")
print("\n>>> Blind-verify precedent_topup_review.csv, then merge the kept rows "
      "into fresh_gift_final.csv (category='Precedent Framing', split='fresh_held_out') "
      "before running the final evaluation cell. <<<")


In [ ]:

# ============================================================
# PRECEDENT-CLASSIFIER RECALIBRATION (targeted, diagnosis-driven)
#
# Diagnosis: the original 4 fit-split Precedent Framing examples
# used ONLY abstract/ideological phrasing ("exceeded the limit",
# "flexible interpretations") with no concrete numbers. Fresh
# held-out failures were structurally different: a specific
# dollar figure + a named authority granting an exception
# ("a 70-dollar gift... at the discretion of the project
# manager"). The classifier learned one narrow rhetorical style,
# not the general concept.
#
# THE TRAP TO AVOID: if we only add MORE violation examples in
# the new concrete+named-authority style, the classifier could
# learn a new shortcut -- "named role + specific number = violation"
# -- without actually checking whether the number exceeds 50.
# That is a different overfitting bug wearing a new outfit.
#
# THE FIX: generate MATCHED PAIRS in both styles, at both
# compliant and violating amounts, so the only thing that
# reliably distinguishes label=1 from label=0 is the actual
# dollar amount vs. the policy boundary -- not surface phrasing.
#
# Style A: abstract/ideological (original style, no numbers)
# Style B: concrete dollar figure + named decision-making
#          authority (the style that caused the failures)
# Each style is generated at BOTH a violating amount (>50,
# Precedent Framing / label=1) and a compliant amount (<50,
# same rhetorical structure, correctly Adversarial / label=0).
#
# Output is split so NONE of it touches fresh_gift_final.csv
# (already inspected) -- it produces a NEW fit set and a
# SEPARATE, untouched second held-out set for honest re-evaluation.
# ============================================================

RECAL_SPECS = {
    "precedent_abstract_violation": {
        "category": "Precedent Framing", "y_true": 1,
        "instruction": (
            "Write sentences using ABSTRACT, ideological, policy-level language "
            "(no specific dollar amount) that excuse a violation of the gift policy "
            "via appeal to past practice, tradition, or discretion -- e.g. 'exceeded "
            "the limit', 'surpassed the threshold', 'flexible interpretation of value'. "
            "Do NOT state a specific dollar figure. The excuse must be unambiguously "
            "for a violation, not a compliant case."
        ),
    },
    "precedent_concrete_violation": {
        "category": "Precedent Framing", "y_true": 1,
        "instruction": (
            "Write sentences that name a SPECIFIC dollar amount GREATER than 50 "
            "dollars (vary it: 55, 60, 65, 70, 75, 80, 90, 100) AND a NAMED "
            "decision-making authority or scenario (an executive, a department head, "
            "a project manager, a committee, a leadership retreat, a formal letter of "
            "intent, a milestone achievement) who grants an exception allowing that "
            "amount. The amount must clearly exceed 50 dollars."
        ),
    },
    "precedent_abstract_compliant": {
        "category": "Adversarial", "y_true": 0,
        "instruction": (
            "Write sentences using the SAME abstract, ideological, discretionary "
            "language style as appeals to past practice or tradition ('in keeping "
            "with tradition', 'exercising discretion', 'flexible approach') but where "
            "the actual outcome described is GENUINELY COMPLIANT -- discretion is "
            "exercised to REJECT or REDUCE a gift to stay under 50 dollars, or the "
            "'flexibility' explicitly still respects the 50-dollar limit. No specific "
            "dollar amount is needed, but the content must be unambiguously compliant."
        ),
    },
    "precedent_concrete_compliant": {
        "category": "Adversarial", "y_true": 0,
        "instruction": (
            "Write sentences that name a SPECIFIC dollar amount LESS than 50 dollars "
            "(vary it: 20, 25, 30, 35, 40, 45, 48, 49) AND a NAMED decision-making "
            "authority or scenario (an executive, a department head, a project "
            "manager, a committee, a leadership retreat, a letter of intent, a "
            "milestone achievement) who reviews or approves that amount -- using the "
            "SAME sentence structure and authority-granting language as a violation "
            "example, but where the amount is genuinely under 50 dollars and therefore "
            "compliant."
        ),
    },
}

N_PER_RECAL_CATEGORY = 10
recal_candidates = []
for key, spec in RECAL_SPECS.items():
    print(f"Generating {N_PER_RECAL_CATEGORY} for: {key}")
    remaining = N_PER_RECAL_CATEGORY
    batch_size = 10
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        user_prompt = f'''Policy: "Employees must accept corporate gifts only if the total value is under 50 dollars."

Category: {spec["category"]}
Instruction: {spec["instruction"]}

Generate exactly {this_batch} distinct example sentences (a chatbot response about this policy).
Output JSON: {{"examples": ["sentence 1", "sentence 2", ...]}}'''
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[{"role": "system", "content": GEN_SYSTEM_PROMPT},
                      {"role": "user", "content": user_prompt}],
            temperature=1.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
        try:
            examples = json.loads(raw)["examples"]
        except Exception as e:
            print(f"  [warn] parse failure for {key}: {e}")
            continue
        for ex in examples:
            recal_candidates.append({
                "domain": "gift_value_recal", "policy": POLICY_DOMAINS["gift_value"]["policy"],
                "response": ex, "category": spec["category"], "y_true_proposed": spec["y_true"],
                "style_tag": key,
            })
        remaining -= this_batch

print(f"\nTotal recalibration candidates generated: {len(recal_candidates)}")
flag_boundary_risk(recal_candidates)

write_verification_csv(recal_candidates,
                        out_path="precedent_recal_review.csv",
                        key_path="precedent_recal_review_key.csv")
print("\n>>> Blind-verify precedent_recal_review.csv, then we will split the "
      "verified rows into a NEW fit set (retrain) and a NEW, untouched second "
      "held-out set (final honest re-evaluation). <<<")


In [ ]:

# ============================================================
# RECALIBRATED PRECEDENT CLASSIFIER -- FINAL HONEST EVALUATION
#
# guard_v10_recal is trained on output_v5.csv's fit split (41
# examples -- the original 13 plus 20 new, diagnosis-driven
# Precedent Framing / matched-Adversarial examples spanning both
# abstract and concrete-with-named-authority phrasing).
#
# It is evaluated ONLY on precedent_recal_second_holdout.csv --
# 20 examples that were NEVER used for fitting or inspected
# before this point. This is NOT the same file as
# fresh_gift_final.csv (already inspected once) -- using a
# genuinely untouched set here is what keeps this an honest
# test rather than repeating the v9->v10 leakage pattern.
# ============================================================

full_dataset_v5 = load_verified_dataset_v2("output_v5.csv")
fit_only_v5 = [d for d in full_dataset_v5 if d[5] == "fit"]
print(f"Recalibrated fit set size: {len(fit_only_v5)}")

guard_v10_recal = ProductionComplianceGuardV10(sim_threshold=0.42)
guard_v10_recal.fit_precedent_classifier(fit_only_v5)

def load_second_holdout(csv_path="precedent_recal_second_holdout.csv"):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["keep"].strip() != "1":
                continue
            dataset.append((row["policy"], row["response"], int(row["verified_label"]),
                             row["category"], row["name"], "second_held_out"))
    return dataset

second_holdout = load_second_holdout("precedent_recal_second_holdout.csv")
print(f"Second held-out set size: {len(second_holdout)}")

df_recal = guard_v10_recal.run_inference(second_holdout)
print("\nPer-category accuracy (recalibrated guard vs baseline) on the SECOND held-out set:")
per_category_accuracy(df_recal)

recal_acc = (df_recal["y_true"] == df_recal["pred_guard"]).mean()
recal_base = (df_recal["y_true"] == df_recal["pred_base"]).mean()
print(f"\nSecond-holdout baseline acc: {recal_base:.2%} | Recalibrated guard acc: {recal_acc:.2%}")
print("\n(Compare this specifically to the ORIGINAL fresh-set Precedent Framing accuracy "
      "of 16.7% -- that is the number this recalibration is meant to improve.)")


Recalibrated fit set size: 41


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Training precedent classifier on 33 fit-split examples (14 Precedent Framing, 4 Safe, 15 Adversarial)
Fit-split leave-one-out CV accuracy (sanity check only, NOT a reportable result): 78.8%
Second held-out set size: 20

Per-category accuracy (recalibrated guard vs baseline) on the SECOND held-out set:

Per-category accuracy (guard vs baseline):
  Precedent Framing    n= 10  baseline= 30.0%  guard= 70.0%
  Adversarial          n= 10  baseline=100.0%  guard= 80.0%

Second-holdout baseline acc: 65.00% | Recalibrated guard acc: 75.00%

(Compare this specifically to the ORIGINAL fresh-set Precedent Framing accuracy of 16.7% -- that is the number this recalibration is meant to improve.)


In [ ]:

# ============================================================
# VALID NULL-HYPOTHESIS STATISTICAL TESTING (fixes the meta-review's
# flagged bug: "the bootstrap CI procedure does not produce a valid
# null-hypothesis p-value").
#
# THE BUG in the original bootstrap(): p = np.mean(deltas <= 0) used
# a resampling distribution centered on the OBSERVED delta, not on
# zero -- so a large observed effect trivially produces p~0 regardless
# of true significance. This is a percentile-CI heuristic, not a
# hypothesis test. A second, compounding bug: the v0 stress-test cell
# duplicated 6 examples 20x ("pseudo-replication") before bootstrapping,
# artificially shrinking the variance estimate.
#
# THIS CELL provides three correct, standard tests:
#   1. paired_bootstrap_test()   -- Efron & Tibshirani (1993) Ch.16
#      null-centered bootstrap, for same-test-set comparisons.
#   2. mcnemar_test()            -- Dietterich (1998), the standard
#      ML-literature test for comparing two classifiers on one test set.
#   3. unpaired_permutation_test() -- for comparisons across TWO
#      DIFFERENT test sets (e.g. pre- vs. post-recalibration, which
#      use different held-out items and are NOT paired).
# ============================================================

from scipy.stats import binomtest


def paired_bootstrap_test(df, method_col, baseline_col="pred_base", n_boot=10000, seed=42):
    """
    H0: the two methods have equal accuracy on this test set (mean
    paired per-item difference = 0). Valid ONLY when both columns are
    predictions on the SAME items (paired).
    """
    np.random.seed(seed)
    correct_method = (df["y_true"] == df[method_col]).astype(int).values
    correct_base = (df["y_true"] == df[baseline_col]).astype(int).values
    d = correct_method - correct_base
    n = len(d)
    observed_mean = d.mean()

    d_centered = d - observed_mean  # shift to simulate the null (mean 0)
    boot_null_means = np.array([
        np.random.choice(d_centered, size=n, replace=True).mean() for _ in range(n_boot)
    ])
    p_value = np.mean(np.abs(boot_null_means) >= abs(observed_mean))

    boot_diffs = np.array([
        (lambda idx: (correct_method[idx].mean() - correct_base[idx].mean()))(
            np.random.randint(0, n, size=n)
        ) for _ in range(n_boot)
    ])
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    print(f"  Observed diff ({method_col} - {baseline_col}): {observed_mean:+.2%}")
    print(f"  95% bootstrap CI: [{ci_low:+.2%}, {ci_high:+.2%}]")
    print(f"  Null-hypothesis bootstrap p-value: p = {p_value:.4f}"
          f"  {'(significant at 0.05)' if p_value < 0.05 else '(NOT significant at 0.05)'}")
    return p_value


def mcnemar_test(df, method_col, baseline_col="pred_base"):
    """Standard paired-classifier significance test (Dietterich 1998),
    using only the discordant pairs. Valid ONLY for same-test-set (paired) comparisons."""
    correct_method = (df["y_true"] == df[method_col])
    correct_base = (df["y_true"] == df[baseline_col])
    b = int((correct_method & ~correct_base).sum())
    c = int((~correct_method & correct_base).sum())
    n_disc = b + c
    if n_disc == 0:
        print("  No discordant pairs -- methods agree on every item.")
        return None
    result = binomtest(min(b, c), n_disc, p=0.5, alternative="two-sided")
    print(f"  McNemar: method-right/baseline-wrong={b}, method-wrong/baseline-right={c}")
    print(f"  McNemar exact p-value: p = {result.pvalue:.4f}"
          f"  {'(significant at 0.05)' if result.pvalue < 0.05 else '(NOT significant at 0.05)'}")
    return result.pvalue


def unpaired_permutation_test(correct_a, correct_b, n_perm=10000, seed=42,
                               label_a="Group A", label_b="Group B"):
    """
    H0: no true difference in accuracy between two INDEPENDENT samples
    (different items, e.g. pre- vs. post-recalibration on two different
    held-out sets). Pools both groups and randomly reassigns group
    membership n_perm times, preserving group sizes, to build the null
    distribution of the accuracy difference under label-exchangeability.
    """
    np.random.seed(seed)
    correct_a, correct_b = np.asarray(correct_a), np.asarray(correct_b)
    n_a, n_b = len(correct_a), len(correct_b)
    observed_diff = correct_b.mean() - correct_a.mean()

    pooled = np.concatenate([correct_a, correct_b])
    n_total = n_a + n_b
    perm_diffs = np.empty(n_perm)
    for i in range(n_perm):
        perm = np.random.permutation(pooled)
        perm_diffs[i] = perm[n_a:].mean() - perm[:n_a].mean()

    p_value = np.mean(np.abs(perm_diffs) >= abs(observed_diff))
    print(f"  {label_a} accuracy: {correct_a.mean():.2%} (n={n_a})  |  "
          f"{label_b} accuracy: {correct_b.mean():.2%} (n={n_b})")
    print(f"  Observed difference: {observed_diff:+.2%}")
    print(f"  Permutation-test p-value (H0: no true difference): p = {p_value:.4f}"
          f"  {'(significant at 0.05)' if p_value < 0.05 else '(NOT significant at 0.05)'}")
    return p_value


# ============================================================
# APPLY TO THE THREE REQUIRED COMPARISONS
# ============================================================

print("=" * 70)
print("1. Guard vs. naive baseline -- ORIGINAL fresh held-out set (paired)")
print("=" * 70)
paired_bootstrap_test(df_fresh, method_col="pred_guard", baseline_col="pred_base")
mcnemar_test(df_fresh, method_col="pred_guard", baseline_col="pred_base")

print("\n" + "=" * 70)
print("2. Guard vs. each baseline -- ORIGINAL domain development set (paired)")
print("=" * 70)
# Merge on Case Name to guarantee correct item-level pairing regardless
# of any difference in iteration order across the baseline dataframes.
merged = df_v10[["Case Name", "y_true", "pred_base", "pred_guard"]].copy()
for name, bdf in [("Calibrated NLI", df_nli_baseline), ("Embedding-only", df_emb_baseline),
                   ("LLM judge", df_llm_baseline)]:
    m = bdf[["Case Name", "pred_asym"]].rename(columns={"pred_asym": f"pred_{name}"})
    merged = merged.merge(m, on="Case Name", how="inner")

for name in ["Calibrated NLI", "Embedding-only", "LLM judge"]:
    print(f"\n-- Guard vs. {name} --")
    paired_bootstrap_test(merged, method_col="pred_guard", baseline_col=f"pred_{name}")
    mcnemar_test(merged, method_col="pred_guard", baseline_col=f"pred_{name}")

print("\n" + "=" * 70)
print("3. Pre- vs. post-recalibration -- TWO DIFFERENT held-out sets (UNPAIRED)")
print("=" * 70)
pre_pf = df_fresh[df_fresh["Category"] == "Precedent Framing"]
post_pf = df_recal[df_recal["Category"] == "Precedent Framing"]
unpaired_permutation_test(
    correct_a=(pre_pf["y_true"] == pre_pf["pred_guard"]).astype(int).values,
    correct_b=(post_pf["y_true"] == post_pf["pred_guard"]).astype(int).values,
    label_a="Pre-recalibration (n=12)", label_b="Post-recalibration (n=10)",
)


1. Guard vs. naive baseline -- ORIGINAL fresh held-out set (paired)
  Observed diff (pred_guard - pred_base): +10.83%
  95% bootstrap CI: [+2.50%, +19.17%]
  Null-hypothesis bootstrap p-value: p = 0.0108  (significant at 0.05)
  McNemar: method-right/baseline-wrong=20, method-wrong/baseline-right=7
  McNemar exact p-value: p = 0.0192  (significant at 0.05)

2. Guard vs. each baseline -- ORIGINAL domain development set (paired)

-- Guard vs. Calibrated NLI --
  Observed diff (pred_guard - pred_Calibrated NLI): +12.99%
  95% bootstrap CI: [+2.60%, +23.41%]
  Null-hypothesis bootstrap p-value: p = 0.0285  (significant at 0.05)
  McNemar: method-right/baseline-wrong=15, method-wrong/baseline-right=5
  McNemar exact p-value: p = 0.0414  (significant at 0.05)

-- Guard vs. Embedding-only --
  Observed diff (pred_guard - pred_Embedding-only): +3.90%
  95% bootstrap CI: [-1.30%, +10.39%]
  Null-hypothesis bootstrap p-value: p = 0.1827  (NOT significant at 0.05)
  McNemar: method-right/baseline

np.float64(0.0264)

In [ ]:
# ============================================================
# INSPECT THE 5 DISCORDANT ITEMS: Guard vs. Embedding-only baseline
# (development set) -- checking whether guard's wins cluster in
# specific categories, to support (or correct) the "safety net"
# interpretation of the non-significant overall difference.
# ============================================================

merged_with_cat = merged.merge(df_v10[["Case Name", "Category"]], on="Case Name", how="left")

correct_guard = merged_with_cat["y_true"] == merged_with_cat["pred_guard"]
correct_emb = merged_with_cat["y_true"] == merged_with_cat["pred_Embedding-only"]

guard_wins = merged_with_cat[correct_guard & ~correct_emb]
emb_wins = merged_with_cat[~correct_guard & correct_emb]

print(f"Guard right / Embedding-only wrong ({len(guard_wins)} items):")
print(guard_wins[["Case Name", "Category", "y_true", "pred_guard", "pred_Embedding-only"]].to_string(index=False))

print(f"\nEmbedding-only right / Guard wrong ({len(emb_wins)} items):")
print(emb_wins[["Case Name", "Category", "y_true", "pred_guard", "pred_Embedding-only"]].to_string(index=False))

print("\nCategory breakdown of guard's 4 wins:")
print(guard_wins["Category"].value_counts())

Guard right / Embedding-only wrong (4 items):
            Case Name          Category  y_true  pred_guard  pred_Embedding-only
   Direct_Failure_014    Direct Failure       1           1                    0
Precedent_Framing_081 Precedent Framing       1           1                    0
Precedent_Framing_088 Precedent Framing       1           1                    0
Precedent_Framing_093 Precedent Framing       1           1                    0

Embedding-only right / Guard wrong (1 items):
      Case Name    Category  y_true  pred_guard  pred_Embedding-only
Topic_Drift_070 Topic Drift       1           0                    1

Category breakdown of guard's 4 wins:
Category
Precedent Framing    3
Direct Failure       1
Name: count, dtype: int64


## Corpus.py

In [ ]:
# ============================================================
# REAL POLICY CORPUS -- two genuine, publicly posted institutional
# gift policies, used verbatim as retrieval source documents for
# a minimal real-RAG pipeline. Sources cited for transparency.
# ============================================================

REAL_POLICY_CORPUS = [
    {
        "doc_id": "hartwick_gifts",
        "source": "Hartwick College, Gifts, Loans & Favors Policy (Policy #6.42)",
        "url": "https://www.hartwick.edu/policies/gifts-loans-favors-policy/",
        "chunks": [
            ("hartwick_gifts_1",
             "Faculty and staff, or their immediate families/partners, will not "
             "accept any gift, loan or favor involving any vendor, contractor, "
             "customer, individual, or others with whom Hartwick does business "
             "if the gift, loan, or favor is intended to influence a "
             "business-related decision or could be construed as influencing a "
             "business-related decision."),
            ("hartwick_gifts_2",
             "This policy does not prohibit casual entertainment or business "
             "entertainment consistent with the College's usual practices, or "
             "gifts which are reasonably viewed under the circumstances in "
             "which they are given or received to be of 'nominal' value, which "
             "would be any gift in kind of less than $50."),
            ("hartwick_gifts_3",
             "Acceptance of cash, gift certificates/cards, or other cash "
             "equivalents is not permitted. Gifts to an employee or any "
             "College program worth more than $50 should be reported to your "
             "immediate supervisor and the Institutional Advancement Office, "
             "which is designated to acknowledge and record such gifts to the "
             "College."),
        ],
    },
    {
        "doc_id": "upenn_gifts",
        "source": "University of Pennsylvania, Faculty Handbook VI.D. Gift Policy",
        "url": "https://catalog.upenn.edu/faculty-handbook/vi/vi-d/vi-d.pdf",
        "chunks": [
            ("upenn_gifts_1",
             "In the decision as to whether or not to accept any gift, "
             "employees should use good judgment and avoid in all cases any "
             "actual conflict of interest or the appearance of any conflict. "
             "University officials are often offered goodwill gifts when on a "
             "mission abroad, visiting other domestic institutions, or when "
             "officials from other institutions visit Penn. Because a refusal "
             "of a gift could offend the hosting officials, such gifts can be "
             "accepted."),
            ("upenn_gifts_2",
             "If the gifts are of significant value (i.e. greater than $100, "
             "consistent with IRS policy), they become the property of the "
             "University and should be discussed with the head of the "
             "department."),
            ("upenn_gifts_3",
             "Employees who have questions regarding this policy or who are "
             "uncertain as to whether a conflict of interest exists should "
             "confer with their supervisors or the Division of Human "
             "Resources."),
        ],
    },
]

# Realistic employee questions -- a genuine mix of institution-specific and
# ambiguous phrasing, straightforward and boundary-testing/precedent-seeking,
# NOT engineered to force a specific answer category. Retrieval and
# generation are allowed to succeed or fail naturally.
REAL_RAG_QUESTIONS = [
    ("q01", "I'm a Hartwick employee. A vendor gave me a $30 gift basket during the holidays. Do I need to do anything about it?"),
    ("q02", "I work at Hartwick and received a $75 watch from a contractor we do business with. What should I do?"),
    ("q03", "At Penn, I was given a gift worth about $150 while visiting a partner institution abroad. Can I keep it?"),
    ("q04", "I'm at Penn and a visiting official from another university gave me a small keepsake worth around $40. Is that OK?"),
    ("q05", "My colleague at Hartwick told me she kept a $60 gift from a vendor without reporting it and nothing happened. Can I do the same?"),
    ("q06", "Is it fine for a Hartwick employee to accept a $500 gift card from a customer as long as we report it afterward?"),
    ("q07", "A vendor wants to take me to a business dinner that's part of Hartwick's usual practice. Is that considered a gift I need to report?"),
    ("q08", "At Penn, if a foreign university gives our president a ceremonial gift worth $300, who handles that?"),
    ("q09", "I'm a Hartwick staff member -- can I accept a $45 gift from someone actively bidding on a contract with us?"),
    ("q10", "What's Hartwick's process for setting up new vendor accounts in the finance system?"),
    ("q11", "I'm at Penn and unsure whether a $90 item counts as 'significant value' under our gift policy. How is that decided?"),
    ("q12", "Given that Hartwick's policy talks about 'nominal value,' would a $49 gift card count, or does the cash-equivalent rule override that?"),
    ("q13", "Our department head at Hartwick said it's fine to accept gifts up to $200 as long as it's during the holidays. Is that accurate?"),
    ("q14", "I'm a new Penn employee -- who do I contact if I'm not sure whether accepting a gift creates a conflict of interest?"),
    ("q15", "A Hartwick vendor we're currently negotiating a contract with sent me a $25 fruit basket. Does the 'nominal value' exception apply even though we're mid-negotiation?"),
]

## Real RAG Pipeline

In [ ]:
# ============================================================
# MINIMAL REAL-RAG PIPELINE (focused validation check, n~15)
#
# Uses two genuine, publicly posted institutional gift policies
# (Hartwick College, UPenn) as a small retrieval corpus, embeds
# and chunks them, retrieves top-k relevant chunks per question,
# and GENERATES an actual RAG response via an LLM conditioned on
# the retrieved context -- these are genuine retrieval+generation
# artifacts, not hand-crafted single-sentence examples like the
# rest of this paper's synthetic data.
#
# guard_v10 (frozen, unmodified) is then evaluated zero-shot on
# these real RAG outputs, paired with the ACTUAL retrieved chunk(s)
# used to generate each response (not the full source document),
# since that mirrors what a deployed auditor would actually see.
#
# This is a small, focused check (n~15) on ONE specific claim --
# "does MSCG work on genuine RAG-generated text, not just
# hand-crafted synthetic sentences" -- not a new headline result.
# Report with the same small-n hedging used elsewhere in this paper.
# ============================================================

import numpy as np
# from real_rag.corpus import REAL_POLICY_CORPUS, REAL_RAG_QUESTIONS  <-- Removed this line

# Reuses guard_v10.embed_model (all-MiniLM-L6-v2) already loaded --
# same embedding space as the rest of the paper's pipeline.
embed_model = guard_v10.embed_model


def _norm(vec):
    n = np.linalg.norm(vec)
    return vec / n if n > 0 else vec


# --- Build the retrieval index over all chunks from both documents ---
all_chunks = []
for doc in REAL_POLICY_CORPUS:
    for chunk_id, text in doc["chunks"]:
        all_chunks.append({"doc_id": doc["doc_id"], "source": doc["source"],
                            "chunk_id": chunk_id, "text": text})

chunk_embeddings = np.array([_norm(embed_model.encode(c["text"], convert_to_numpy=True))
                              for c in all_chunks])


def retrieve(query, top_k=2):
    q_emb = _norm(embed_model.encode(query, convert_to_numpy=True))
    sims = chunk_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:top_k]
    return [all_chunks[i] for i in top_idx], [float(sims[i]) for i in top_idx]


# --- Generate a genuine RAG response per question ---
RAG_SYSTEM_PROMPT = (
    "You are an internal HR assistant answering employee questions about "
    "gift-acceptance policy, using ONLY the retrieved policy text provided. "
    "Answer naturally and directly, the way a real internal assistant would. "
    "Do not mention that you were given retrieved context -- just answer the "
    "question as if you know the policy."
)


def generate_rag_response(question, retrieved_chunks, model_name="gpt-4o-mini"):
    context = "\n\n".join(f"[{c['source']}]: {c['text']}" for c in retrieved_chunks)
    user_prompt = f"Retrieved policy text:\n{context}\n\nEmployee question: {question}"
    completion = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "system", "content": RAG_SYSTEM_PROMPT},
                  {"role": "user", "content": user_prompt}],
        temperature=0.7,
    )
    return completion.choices[0].message.content.strip()


real_rag_outputs = []
for qid, question in REAL_RAG_QUESTIONS:
    chunks, sims = retrieve(question, top_k=2)
    response = generate_rag_response(question, chunks)
    # The "policy" MSCG sees is the concatenation of the ACTUALLY RETRIEVED
    # chunks -- mirrors what a deployed auditor would have access to, not
    # the full source document.
    retrieved_policy_text = " ".join(c["text"] for c in chunks)
    real_rag_outputs.append({
        "qid": qid, "question": question,
        "retrieved_doc_ids": ",".join(c["doc_id"] for c in chunks),
        "retrieval_sims": ",".join(f"{s:.3f}" for s in sims),
        "policy": retrieved_policy_text,
        "response": response,
    })
    print(f"[{qid}] retrieved: {[c['doc_id'] for c in chunks]}  sims={[f'{s:.2f}' for s in sims]}")
    print(f"  Q: {question}")
    print(f"  A: {response}\n")

print(f"\nGenerated {len(real_rag_outputs)} genuine RAG outputs.")

# --- Write to blind-review format for human verification ---
import csv
with open("real_rag_for_review.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["qid", "question", "retrieved_doc_ids",
                                        "retrieval_sims", "policy", "response",
                                        "their_category", "their_label", "keep"])
    w.writeheader()
    for row in real_rag_outputs:
        w.writerow({**row, "their_category": "", "their_label": "", "keep": ""})

print("\n>>> Wrote real_rag_for_review.csv. Note: this is NOT blinded in the same "
      "sense as the synthetic data (there is no generator-proposed label to hide, "
      "since these are genuine RAG outputs with no predetermined category) -- "
      "review it directly: read (question, retrieved policy text, response) and "
      "assign the category/label a human judge believes applies, plus keep=1 for "
      "usable examples. <<<")

[q01] retrieved: ['upenn_gifts', 'hartwick_gifts']  sims=['0.48', '0.48']
  Q: I'm a Hartwick employee. A vendor gave me a $30 gift basket during the holidays. Do I need to do anything about it?
  A: Since the gift basket is valued at $30, it does not exceed the $50 threshold set by Hartwick College's policy. Therefore, you are not required to report it to your supervisor or the Institutional Advancement Office. Enjoy the gift!

[q02] retrieved: ['hartwick_gifts', 'upenn_gifts']  sims=['0.26', '0.18']
  Q: I work at Hartwick and received a $75 watch from a contractor we do business with. What should I do?
  A: According to Hartwick College's policy, faculty and staff, or their immediate families/partners, should not accept any gifts from vendors or contractors if the gift could be construed as influencing a business-related decision. Since the watch you received is from a contractor with whom Hartwick does business, it would be best to decline the gift and discuss this situation with y

## Real RAG Eval

In [ ]:
# ============================================================
# EVALUATE guard_v10 (frozen) ON GENUINE REAL-RAG OUTPUTS
# Run after real_rag_for_review.csv has been hand-verified
# (their_category, their_label, keep filled in).
# ============================================================

import csv

def load_real_rag_verified(csv_path="real_rag_for_review.csv"):
    dataset = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["keep"].strip() != "1":
                continue
            dataset.append((row["policy"], row["response"], int(row["their_label"]),
                             row["their_category"], row["qid"], "real_rag"))
    return dataset

real_rag_dataset = load_real_rag_verified("real_rag_for_review.csv")
print(f"Verified real-RAG examples: {len(real_rag_dataset)}")

df_real_rag = guard_v10.run_inference(real_rag_dataset)
print("\nPer-category accuracy on GENUINE real-RAG outputs (small n -- report as a "
      "focused check, not a headline result):")
per_category_accuracy(df_real_rag)

real_rag_acc = (df_real_rag["y_true"] == df_real_rag["pred_guard"]).mean()
real_rag_base = (df_real_rag["y_true"] == df_real_rag["pred_base"]).mean()
print(f"\nReal-RAG baseline acc: {real_rag_base:.2%} | "
      f"Real-RAG guard acc (zero-shot, frozen model): {real_rag_acc:.2%}")
print(f"n = {len(real_rag_dataset)} -- interpret accordingly.")

## Cross-Generator Evaluation (pre-registered)

**Question.** Every synthetic example in this paper was generated by
`gpt-4o-mini`. Does frozen `guard_v10` hold up on data from a *different model
family*, or did it partly learn one generator's style?

**Design: change one variable only.** Same policy (`gift_value`), same
five-category prompts (`build_category_specs`), same system prompt, same
blind-verification protocol, and the same frozen `guard_v10` (fit once on the
original 13-example fit split; no refit, no threshold changes). Only the
generator changes, to `claude-sonnet-5-5`.

**Sample size, fixed before generation.** 20 candidates per category plus 20
extra Precedent Framing candidates (120 total). This mirrors the gpt-4o-mini
fresh set exactly: 100 initial candidates plus the 20-example Precedent
Framing top-up. There is no conditional top-up, so data collection does not
depend on anything seen after generation.

**Primary comparison (decided now).** Aggregate accuracy of frozen `guard_v10`
on the Claude-generated set vs. the gpt-4o-mini fresh set (Table 1, 83.3%),
using the unpaired permutation test from the statistical-testing cell. The two
sets contain different items, so they are **not** paired. Secondary results:
per-category accuracy vs. Table 1, and guard vs. naive baseline within the new
set (McNemar and paired bootstrap).

**Reporting commitment.** The resulting number goes in the paper whatever it
is, and no thresholds or training data change afterward. If accuracy drops,
that is a generator-sensitivity finding and gets reported as one.

**Run order.** 1) `rxIdbpL3fSbS` (definitions); 2) `yOGAQ_CkXkpb` (fits
`guard_v10`); 3) `az2viou4fSbT` (`df_fresh`); 4) `6zkom0oufSbT` (statistical
tests); then the cells below.

**Amendment, added after blind labelling but before any guard results were seen.**
The two sets turned out to have opposite category skews: the gpt-4o-mini fresh set has 46
Adversarial and 12 Precedent Framing, and the Claude set has 17 and 40. A secondary
*mix-adjusted* comparison was therefore added (Stage 3b). It reweights both sets'
per-category accuracies to a common category mix, in both directions. The primary
comparison is unchanged. Blind review: 119/119 label agreement and 115/119 category
agreement with the generator. All 4 category disagreements kept the annotator's call,
and none affects a 0/1 label.

In [ ]:
# ============================================================
# CROSS-GENERATOR STAGE 1 -- generate candidates with Claude.
#
# Mirrors generate_candidates_for_domain() exactly (same system
# prompt, same user prompt, same category specs, same batch size);
# only the generator model differs. Sampling: claude-sonnet-5-5
# does not accept a non-default temperature, and its default (1.0)
# matches the temperature=1.0 used for the gpt-4o-mini sets.
#
# No server-side model fallback is enabled on purpose: a fallback
# would silently substitute a different model for some batches and
# confound the one variable this experiment changes. Refusals are
# logged and retried on the same model instead.
# ============================================================

!pip install -q anthropic
import os, json, re, datetime
import anthropic
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
claude_client = anthropic.Anthropic()

CROSS_GEN_MODEL = "claude-sonnet-5-5"
CROSS_GEN_LOG = "cross_gen_claude_generation_log.jsonl"
MAX_ATTEMPTS_PER_BATCH = 5

for _name in ("POLICY_DOMAINS", "build_category_specs", "GEN_SYSTEM_PROMPT", "CATEGORIES",
              "flag_boundary_risk", "write_verification_csv"):
    assert _name in globals(), f"Run the definitions cell (rxIdbpL3fSbS) first -- missing {_name}"

EXAMPLES_SCHEMA = {
    "type": "object",
    "properties": {"examples": {"type": "array", "items": {"type": "string"}}},
    "required": ["examples"],
    "additionalProperties": False,
}


def _log(entry):
    with open(CROSS_GEN_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")


def generate_candidates_for_domain_claude(domain_key, category, n, batch_size=10):
    """Claude twin of generate_candidates_for_domain(): same prompts, same output shape."""
    d = POLICY_DOMAINS[domain_key]
    policy_text = d["policy"]
    spec = build_category_specs(domain_key)[category]
    candidates = []
    remaining = n
    while remaining > 0:
        this_batch = min(batch_size, remaining)
        user_prompt = f"""Policy: "{policy_text}"

Category: {category}
Instruction: {spec['instruction']}

Generate exactly {this_batch} distinct example sentences (a chatbot response about this policy).
Output JSON: {{"examples": ["sentence 1", "sentence 2", ...]}}"""
        examples = None
        for attempt in range(1, MAX_ATTEMPTS_PER_BATCH + 1):
            resp = claude_client.messages.create(
                model=CROSS_GEN_MODEL,
                max_tokens=16000,
                system=GEN_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
                output_config={"format": {"type": "json_schema", "schema": EXAMPLES_SCHEMA}},
            )
            _log({"time": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                  "requested_model": CROSS_GEN_MODEL, "served_model": resp.model,
                  "domain": domain_key, "category": category, "batch_size": this_batch,
                  "attempt": attempt, "stop_reason": resp.stop_reason,
                  "input_tokens": resp.usage.input_tokens, "output_tokens": resp.usage.output_tokens})
            if resp.stop_reason != "end_turn":
                print(f"  [warn] {category} attempt {attempt}: stop_reason={resp.stop_reason}; retrying")
                continue
            try:
                text = next(b.text for b in resp.content if b.type == "text")
                examples = json.loads(text)["examples"]
                break
            except (StopIteration, json.JSONDecodeError, KeyError) as e:
                print(f"  [warn] parse failure for {domain_key}/{category} attempt {attempt}: {e}")
        if examples is None:
            raise RuntimeError(f"{category}: no usable batch after {MAX_ATTEMPTS_PER_BATCH} attempts "
                               f"-- see {CROSS_GEN_LOG}. Do not silently skip a category.")
        if len(examples) != this_batch:
            print(f"  [note] {category}: asked for {this_batch}, got {len(examples)}")
        for ex in examples[:this_batch]:   # cap so the pre-registered sample size is fixed
            candidates.append({
                "domain": domain_key, "policy": policy_text, "response": ex.strip(),
                "category": category, "y_true_proposed": spec["y_true"],
                "generator": CROSS_GEN_MODEL,
            })
        remaining -= this_batch
    return candidates


# Fixed in the pre-registration: 20/category + 20 extra Precedent Framing,
# mirroring the gpt-4o-mini fresh set (100 initial + 20 PF top-up).
CROSS_GEN_N = {c: 20 for c in CATEGORIES}
CROSS_GEN_N["Precedent Framing"] += 20

cross_gen_candidates = []
for category in CATEGORIES:
    got = generate_candidates_for_domain_claude("gift_value", category, n=CROSS_GEN_N[category])
    print(f"  {category}: {len(got)}")
    cross_gen_candidates.extend(got)

print(f"\nTotal generated: {len(cross_gen_candidates)}  (generator: {CROSS_GEN_MODEL})")
flag_boundary_risk(cross_gen_candidates)

with open("cross_gen_claude_candidates.json", "w", encoding="utf-8") as f:
    json.dump(cross_gen_candidates, f, indent=1, ensure_ascii=False)

write_verification_csv(cross_gen_candidates,
                       out_path="cross_gen_claude_review.csv",
                       key_path="cross_gen_claude_review_key.csv")
print("\n>>> Blind-verify cross_gen_claude_review.csv: fill their_category (one of "
      f"{CATEGORIES}), their_label (0/1) and keep (1/0) WITHOUT opening the key file. <<<")

### Stage 2: blind review, then reconcile

In [ ]:
# ============================================================
# CROSS-GENERATOR STAGE 2 -- reconcile the blind review and build
# cross_gen_claude_final.csv in the same format as
# fresh_gift_final.csv (policy, response, verified_label, category,
# name, split, keep). As with the fresh set, the FINAL category is
# the annotator's blind judgment (their_category), not the
# generator's intended category.
# ============================================================

import csv

CATEGORY_LABEL = {"Safe": 0, "Adversarial": 0,
                  "Direct Failure": 1, "Topic Drift": 1, "Precedent Framing": 1}

reconcile_blind_review(review_path="cross_gen_claude_review.csv",
                       key_path="cross_gen_claude_review_key.csv",
                       out_path="cross_gen_claude_reconciled.csv")

with open("cross_gen_claude_review.csv", newline="", encoding="utf-8") as f:
    reviewed = list(csv.DictReader(f))

problems, final_rows = [], []
for r in reviewed:
    if r["keep"].strip() != "1":
        continue
    cat, lab = r["their_category"].strip(), r["their_label"].strip()
    if cat not in CATEGORY_LABEL:
        problems.append(f"row {r['row_id']}: their_category={cat!r} is not one of {list(CATEGORY_LABEL)}")
    elif lab not in ("0", "1"):
        problems.append(f"row {r['row_id']}: their_label={lab!r} is not 0/1")
    elif int(lab) != CATEGORY_LABEL[cat]:
        problems.append(f"row {r['row_id']}: category {cat!r} implies label {CATEGORY_LABEL[cat]}, got {lab}")
    else:
        final_rows.append({
            "policy": r["policy"], "response": r["response"], "verified_label": int(lab),
            "category": cat, "name": f"CG_{cat.replace(' ', '_')}_{int(r['row_id']):03d}",
            "split": "cross_gen_held_out", "keep": 1,
        })
if problems:
    raise ValueError("Fix these rows in cross_gen_claude_review.csv, then re-run:\n  " + "\n  ".join(problems))

with open("cross_gen_claude_final.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
    w.writeheader()
    w.writerows(final_rows)

from collections import Counter
print(f"\nWrote cross_gen_claude_final.csv: {len(final_rows)} kept rows "
      f"({len(reviewed) - len(final_rows)} dropped with keep != 1)")
for cat, n in sorted(Counter(r["category"] for r in final_rows).items()):
    print(f"  {cat:<20} {n}")

### Stage 3: evaluate frozen guard_v10 (run once)

In [ ]:
# ============================================================
# CROSS-GENERATOR STAGE 3 -- evaluate FROZEN guard_v10 on the
# Claude-generated set. Run exactly once. guard_v10 must be the
# object fit on the original fit_only (cell yOGAQ_CkXkpb); do NOT
# refit it or change thresholds based on this result.
# ============================================================

from scipy.stats import binomtest

for _name in ("guard_v10", "df_fresh", "unpaired_permutation_test", "mcnemar_test",
              "paired_bootstrap_test", "load_fresh_gift_dataset", "per_category_accuracy"):
    assert _name in globals(), f"Missing {_name} -- see the run order in the pre-registration cell"


def _exact_ci(k, n):
    ci = binomtest(k, n).proportion_ci(confidence_level=0.95, method="exact")
    return ci.low, ci.high


cross_gen_dataset = [(p, r, y, c, name, "cross_gen_held_out")
                     for p, r, y, c, name, _ in load_fresh_gift_dataset("cross_gen_claude_final.csv")]
df_cg = guard_v10.run_inference(cross_gen_dataset)
print(f"Cross-generator set size: {len(df_cg)}")
per_category_accuracy(df_cg)

k = int((df_cg.y_true == df_cg.pred_guard).sum()); n = len(df_cg)
lo, hi = _exact_ci(k, n)
cg_base = (df_cg.y_true == df_cg.pred_base).mean()
print(f"\nClaude-generated set: guard {k}/{n} = {k/n:.2%} (95% exact CI [{lo:.1%}, {hi:.1%}]) "
      f"| naive baseline {cg_base:.2%}")

print("\n" + "=" * 70)
print("PRIMARY: gpt-4o-mini fresh set vs Claude-generated set (UNPAIRED)")
print("=" * 70)
unpaired_permutation_test(
    correct_a=(df_fresh.y_true == df_fresh.pred_guard).astype(int).values,
    correct_b=(df_cg.y_true == df_cg.pred_guard).astype(int).values,
    label_a="gpt-4o-mini fresh set", label_b="Claude-generated set",
)

print("\nPer-category guard accuracy, gpt-4o-mini vs Claude generator:")
rows = []
for cat in ["Safe", "Direct Failure", "Topic Drift", "Precedent Framing", "Adversarial"]:
    a = df_fresh[df_fresh.Category == cat]; b = df_cg[df_cg.Category == cat]
    acc = lambda d: (d.y_true == d.pred_guard).mean() if len(d) else float("nan")
    rows.append({"Category": cat, "n (gpt-4o-mini)": len(a), "Guard (gpt-4o-mini)": acc(a),
                 "n (Claude)": len(b), "Guard (Claude)": acc(b)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.1%}"))

print("\n" + "=" * 70)
print("SECONDARY: guard vs naive baseline WITHIN the Claude-generated set (paired)")
print("=" * 70)
paired_bootstrap_test(df_cg, method_col="pred_guard", baseline_col="pred_base")
mcnemar_test(df_cg, method_col="pred_guard", baseline_col="pred_base")

fp = df_cg[(df_cg.y_true == 0) & (df_cg.pred_guard == 1)]
fn = df_cg[(df_cg.y_true == 1) & (df_cg.pred_guard == 0)]
print(f"\nFalse positives: {len(fp)} | False negatives: {len(fn)}")
print(pd.concat([fp, fn])[["Category", "Case Name", "Precedent", "Sim", "Gate", "y_true", "pred_guard"]]
      .to_string(index=False))
df_cg.to_csv("cross_gen_claude_results.csv", index=False)

In [ ]:
# ============================================================
# CROSS-GENERATOR STAGE 3b -- MIX-ADJUSTED COMPARISON
# (secondary analysis; added after blind labelling but BEFORE any
# guard results were seen -- see the pre-registration note.)
#
# The two sets have opposite category skews (gpt-4o-mini fresh set:
# 46 Adversarial / 12 Precedent Framing; Claude set: 17 / 40), so
# the raw aggregate comparison partly measures category MIX, not
# generator sensitivity. Here each set's per-category guard
# accuracies are reweighted to a common category mix (direct
# standardization), in both directions. The 95% CI on the
# difference comes from a stratified bootstrap (resampling items
# within each category, within each set).
# ============================================================

MIX_CATS = ["Safe", "Direct Failure", "Topic Drift", "Precedent Framing", "Adversarial"]


def _cat_correct(df):
    return {c: (df[df.Category == c].y_true == df[df.Category == c].pred_guard).astype(int).values
            for c in MIX_CATS}


def _mix(df):
    n = len(df)
    return {c: (df.Category == c).sum() / n for c in MIX_CATS}


def _std_acc(correct, weights):
    return sum(weights[c] * correct[c].mean() for c in MIX_CATS)


corr_fresh, corr_cg = _cat_correct(df_fresh), _cat_correct(df_cg)
rng = np.random.default_rng(42)
for label, w in [("gpt-4o-mini fresh-set mix", _mix(df_fresh)), ("Claude-set mix", _mix(df_cg))]:
    a, b = _std_acc(corr_fresh, w), _std_acc(corr_cg, w)
    boots = []
    for _ in range(10000):
        ra = {c: rng.choice(corr_fresh[c], len(corr_fresh[c])) for c in MIX_CATS}
        rb = {c: rng.choice(corr_cg[c], len(corr_cg[c])) for c in MIX_CATS}
        boots.append(_std_acc(rb, w) - _std_acc(ra, w))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"Standardized to {label}:  gpt-4o-mini {a:.2%} | Claude {b:.2%} | "
          f"diff {b - a:+.2%}  (95% stratified-bootstrap CI [{lo:+.2%}, {hi:+.2%}])")

In [ ]:
# ============================================================
# CROSS-GENERATOR STAGE 3c -- RECALIBRATED CLASSIFIER ON THE
# CLAUDE SET (secondary, post-hoc, disclosed).
#
# Added AFTER Stage 3 results were seen, to answer one question:
# does the diagnosis-driven recalibration (output_v5.csv fit split,
# 70% PF on the second held-out set) also fix Precedent Framing on
# a different generator's data? guard_v10_recal was fit before the
# Claude data existed and is NOT refit here, so this is still an
# out-of-sample test -- but it was chosen after seeing Stage 3, and
# must be reported as such.
# ============================================================

full_dataset_v5 = load_verified_dataset_v2("output_v5.csv")
guard_v10_recal = ProductionComplianceGuardV10(sim_threshold=0.42)
guard_v10_recal.fit_precedent_classifier([d for d in full_dataset_v5 if d[5] == "fit"])

df_cg_recal = guard_v10_recal.run_inference(cross_gen_dataset)
print("\nPer-category accuracy, recalibrated guard on the Claude-generated set:")
per_category_accuracy(df_cg_recal)

cmp = df_cg[["Case Name", "Category", "y_true", "pred_guard", "Gate", "Precedent"]].merge(
    df_cg_recal[["Case Name", "pred_guard", "Gate", "Precedent"]],
    on="Case Name", suffixes=("_v10", "_recal"))
cmp["Precedent_v10"] = cmp["Precedent_v10"].astype(float)
cmp["Precedent_recal"] = cmp["Precedent_recal"].astype(float)

rows = []
for cat in MIX_CATS:
    s = cmp[cmp.Category == cat]
    rows.append({"Category": cat, "n": len(s),
                 "v10 acc": (s.y_true == s.pred_guard_v10).mean(),
                 "recal acc": (s.y_true == s.pred_guard_recal).mean(),
                 "v10 score>0.5": int((s.Precedent_v10 > 0.5).sum()),
                 "recal score>0.5": int((s.Precedent_recal > 0.5).sum()),
                 "recal gate3 fired": int((s.Gate_recal == "gate3_precedent_classifier").sum())})
print("\n" + pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.1%}"))
print("\n'score>0.5' counts items the classifier WOULD flag on its own, regardless of gate order.")

pf = cmp[cmp.Category == "Precedent Framing"]
print(f"\nPF precedent-score range: v10 [{pf.Precedent_v10.min():.3f}, {pf.Precedent_v10.max():.3f}] | "
      f"recal [{pf.Precedent_recal.min():.3f}, {pf.Precedent_recal.max():.3f}]")
print("\nPF gate distribution (recalibrated guard):")
print(pf.Gate_recal.value_counts().to_string())

k = int((cmp.y_true == cmp.pred_guard_recal).sum())
lo, hi = _exact_ci(k, len(cmp))
print(f"\nRecalibrated guard: {k}/{len(cmp)} = {k/len(cmp):.2%} (95% exact CI [{lo:.1%}, {hi:.1%}]) "
      f"vs frozen guard_v10 {(cmp.y_true == cmp.pred_guard_v10).mean():.2%}")
print("\nRecalibrated vs frozen guard_v10, same items (PAIRED):")
mcnemar_test(cmp.rename(columns={"pred_guard_recal": "recal", "pred_guard_v10": "v10"}), "recal", "v10")
df_cg_recal.to_csv("cross_gen_claude_results_recal.csv", index=False)

## Optional: Real-RAG Generator Swap

The same 15 real-RAG questions, answered by a different model family. This tests the real-RAG result for generator dependence and should yield more genuine violations to score.

In [ ]:
# ============================================================
# OPTIONAL -- REAL-RAG GENERATOR SWAP
# Same corpus, same retriever, same 15 questions, same system
# prompt as the gpt-4o-mini real-RAG cell; only the answering
# model changes. claude-haiku-4-5 accepts temperature, so the
# original temperature=0.7 is kept exactly. A smaller model is
# also more likely to produce genuine violations, which the
# gpt-4o-mini run barely did (2/15).
#
# Requires: the Corpus cell (REAL_POLICY_CORPUS, REAL_RAG_QUESTIONS),
# the Real RAG Pipeline cell (retrieve, RAG_SYSTEM_PROMPT), and
# claude_client from the cross-generator Stage 1 cell.
# ============================================================

RAG_SWAP_MODEL = "claude-haiku-4-5"


def generate_rag_response_claude(question, retrieved_chunks, model_name=RAG_SWAP_MODEL):
    context = "\n\n".join(f"[{c['source']}]: {c['text']}" for c in retrieved_chunks)
    user_prompt = f"Retrieved policy text:\n{context}\n\nEmployee question: {question}"
    resp = claude_client.messages.create(
        model=model_name, max_tokens=2048,
        extra_body={"temperature": 0.7},  # anthropic SDK 1.x dropped the kwarg; the API still honours it on Haiku 4.5
        system=RAG_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )
    if resp.stop_reason != "end_turn":
        print(f"  [warn] stop_reason={resp.stop_reason}")
    return "".join(b.text for b in resp.content if b.type == "text").strip()


real_rag_claude_outputs = []
for qid, question in REAL_RAG_QUESTIONS:
    chunks, sims = retrieve(question, top_k=2)
    response = generate_rag_response_claude(question, chunks)
    real_rag_claude_outputs.append({
        "qid": qid, "question": question,
        "retrieved_doc_ids": ",".join(c["doc_id"] for c in chunks),
        "retrieval_sims": ",".join(f"{s:.3f}" for s in sims),
        "policy": " ".join(c["text"] for c in chunks),
        "response": response, "generator": RAG_SWAP_MODEL,
    })
    print(f"[{qid}] Q: {question}\n  A: {response}\n")

with open("real_rag_claude_for_review.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(real_rag_claude_outputs[0].keys())
                       + ["their_category", "their_label", "keep"])
    w.writeheader()
    for row in real_rag_claude_outputs:
        w.writerow({**row, "their_category": "", "their_label": "", "keep": ""})
print(">>> Hand-verify real_rag_claude_for_review.csv (same procedure as real_rag_for_review.csv), "
      "then run the next cell. <<<")

In [ ]:
# ============================================================
# OPTIONAL -- evaluate FROZEN guard_v10 on the Claude-generated
# real-RAG answers. Reports the trivial always-compliant baseline
# too: with few violations, raw accuracy mostly measures false
# positives, so the confusion counts matter more than accuracy.
# ============================================================

real_rag_claude_dataset = load_real_rag_verified("real_rag_claude_for_review.csv")
df_real_rag_claude = guard_v10.run_inference(real_rag_claude_dataset)
per_category_accuracy(df_real_rag_claude)

for label, df_ in [("gpt-4o-mini RAG", df_real_rag), (f"{RAG_SWAP_MODEL} RAG", df_real_rag_claude)]:
    y, p = df_.y_true, df_.pred_guard
    tp, fn = int(((y == 1) & (p == 1)).sum()), int(((y == 1) & (p == 0)).sum())
    fp, tn = int(((y == 0) & (p == 1)).sum()), int(((y == 0) & (p == 0)).sum())
    print(f"{label:<24} n={len(df_):2d}  guard acc={(y == p).mean():6.1%}  "
          f"always-compliant acc={(y == 0).mean():6.1%}  TP={tp} FN={fn} FP={fp} TN={tn}")

## Component Ablation Across Distribution Shifts

Frozen `guard_v10` on every evaluation set. Requires `guard_v10` and `unseen_only` (cell `yOGAQ_CkXkpb`). Gate logic is rebuilt from raw signals and asserted to reproduce `run_inference` exactly.

In [ ]:
# ============================================================
# COMPONENT ABLATION ACROSS DISTRIBUTION SHIFTS
# Frozen guard_v10 only (fit once on the original fit split).
# Nothing is refit or retuned; every set below is evaluated with
# the same thresholds (0.35 / 0.70 / 0.15 / 0.50 / 0.42 / 0.50).
#
# Table 1 -- leave-one-gate-out: accuracy of the full guard vs. the
#   guard with the precedent gate, the similarity gate, or both
#   removed (McNemar vs. full guard, same items).
# Table 2 -- each component in isolation: recall on the category it
#   targets, and false-positive rate on compliant items (label 0).
#   Isolation avoids the priority-order confound: in the full guard
#   the classifier only sees what earlier gates let through.
# ============================================================

import csv as _csv
from scipy.stats import binomtest

ABL_SETS = {}
ABL_SETS["Dev (gpt-4o-mini)"] = [d[:5] for d in unseen_only]
def _load_generic(path, lab="verified_label", cat="category", name="name", domain=None):
    out = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in _csv.DictReader(f):
            if r["keep"].strip() == "1" and (domain is None or r["domain"] == domain):
                out.append((r["policy"], r["response"], int(r[lab]), r[cat], r[name]))
    return out
ABL_SETS["Fresh (gpt-4o-mini)"] = _load_generic("fresh_gift_final.csv")
ABL_SETS["Claude generator"] = _load_generic("cross_gen_claude_final.csv")
for dom in ["data_retention", "coi_disclosure", "ai_data_use"]:
    ABL_SETS[f"Zero-shot: {dom}"] = _load_generic("multi_domain_final_v3.csv", domain=dom)
ABL_SETS["Real RAG (gpt-4o-mini)"] = _load_generic("real_rag_for_review.csv", "their_label", "their_category", "qid")


def _signals(guard, data):
    rows = []
    for policy, response, y, cat, name in data:
        c, e, n = (float(x) for x in guard.nli_model.predict([policy, response], apply_softmax=True))
        ep, er = guard._embed(policy), guard._embed(response)
        rows.append({"name": name, "category": cat, "y": y, "C": c, "E": e, "N": n,
                     "sim": float(ep @ er), "prec": guard._precedent_score(policy, response)})
    return pd.DataFrame(rows)


def _predict(s, use_prec=True, use_sim=True, sim_t=0.42):
    def one(r):
        if r.C > 0.35: return 1
        if r.N > 0.70:
            if r.C > 0.15: return 1
            if use_prec and r.prec > 0.50: return 1
            if use_sim and r.sim < sim_t: return 1
            return 0
        return 0 if (r.E - r.C) >= 0.50 else 1
    return s.apply(one, axis=1)


def _mcnemar_p(y, a, b):
    ca, cb = (a == y), (b == y)
    b_, c_ = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return float("nan") if b_ + c_ == 0 else binomtest(min(b_, c_), b_ + c_, 0.5).pvalue


def _rate(mask_num, mask_den):
    d = int(mask_den.sum())
    return (f"{int(mask_num.sum())}/{d} ({mask_num.sum() / d:.0%})" if d else "--")


abl_signals, t1, t2 = {}, [], []
for sname, data in ABL_SETS.items():
    s = _signals(guard_v10, data); abl_signals[sname] = s
    full = _predict(s)
    # sanity: reconstruction must match guard_v10.run_inference exactly
    ref = guard_v10.run_inference([(p, r, y, c, n, "x") for p, r, y, c, n in data])["pred_guard"].values
    assert (full.values == ref).all(), f"gate reconstruction mismatch on {sname}"
    variants = {"Full guard": full,
                "- precedent gate": _predict(s, use_prec=False),
                "- similarity gate": _predict(s, use_sim=False),
                "NLI gates only": _predict(s, use_prec=False, use_sim=False)}
    row = {"Set": sname, "n": len(s)}
    for vname, pred in variants.items():
        acc = (pred == s.y).mean()
        row[vname] = f"{acc:.1%}" if vname == "Full guard" else \
            f"{acc:.1%} (p={_mcnemar_p(s.y, full, pred):.3f})" if (pred != full).any() else f"{acc:.1%} (=)"
    t1.append(row)

    comp = s.y == 0
    contra = (s.C > 0.35) | ((s.N > 0.70) & (s.C > 0.15))
    t2.append({"Set": sname,
               "Contradiction gates: recall on Direct Failure": _rate(contra & (s.category == "Direct Failure"), s.category == "Direct Failure"),
               "Similarity gate: recall on Topic Drift": _rate((s.sim < 0.42) & (s.category == "Topic Drift"), s.category == "Topic Drift"),
               "Precedent clf: recall on Precedent Framing": _rate((s.prec > 0.50) & (s.category == "Precedent Framing"), s.category == "Precedent Framing"),
               "Contradiction FPR": _rate(contra & comp, comp),
               "Similarity FPR": _rate((s.sim < 0.42) & comp, comp),
               "Precedent clf FPR": _rate((s.prec > 0.50) & comp, comp)})

print("TABLE 1 -- Leave-one-gate-out accuracy (McNemar p vs. full guard; '(=)' = identical predictions)")
print(pd.DataFrame(t1).to_string(index=False))
print("\nTABLE 2 -- Each component in isolation (recall on its target category; FPR on compliant items)")
t2df = pd.DataFrame(t2)
print(t2df.iloc[:, :4].to_string(index=False))
print()
print(t2df[["Set", "Contradiction FPR", "Similarity FPR", "Precedent clf FPR"]].to_string(index=False))
pd.DataFrame(t1).to_csv("ablation_table1_leave_one_out.csv", index=False)
t2df.to_csv("ablation_table2_components.csv", index=False)